# Podnájom alebo hypotéka? – jeden balík
Notebook obsahuje všetko, čo projekt potrebuje – **dáta, kód modelu, váhy neurónovej siete** – a nič sa v ňom nesťahuje ani neinštaluje (stačí Google Colab: *Runtime → Run all*).

Čo robí:
1. **Majetok v oboch scenároch** pre zvolený počiatočný rok (časť A),
2. **rada modelu** pre zadanú situáciu (časť B),
3. **vyhodnotenie na histórii**: pre každý počiatočný rok odpoveď modelu len z dát do tohto roku vs. skutočný víťaz (časť C).

Dáta: Španielsko 1870–2025, ročne (JST Macrohistory do 2020, 2021–2025 predĺženie z ECB/BIS). Zdroje a slabiny: https://github.com/adibuj2804/rent-vs-buy/blob/main/data/SOURCES.md · správa: https://adibuj2804.github.io/rent-vs-buy/sprava.html

*Vzdelávací model, nie finančné poradenstvo.*

## 0 · Dáta, model, sieť (spusti bez zmien)

In [ ]:
# DÁTA: ročné rady pre Španielsko (vložené; stĺpec `source` hovorí, čo je JST a čo predĺženie)
import io, json
import numpy as np
import pandas as pd
DATA_CSV = '''year,iso,cpi,ltrate,housing_capgain,housing_rent_yd,eq_tr,source
1870,ESP,0.2823471260174294,11.194029850746269,,,,JST
1871,ESP,0.2830402048773203,9.917355371900827,,,,JST
1872,ESP,0.2914561624617089,11.560693641618498,,,,JST
1873,ESP,0.3141627460619417,22.607385079125848,,,,JST
1874,ESP,0.3103673142101587,23.346303501945524,,,,JST
1875,ESP,0.3036675518978807,17.59530791788856,,,,JST
1876,ESP,0.2995420824937686,23.715415019762844,,,,JST
1877,ESP,0.3024134091990306,22.988505747126435,,,,JST
1878,ESP,0.300994247724016,19.672131147540984,,,,JST
1879,ESP,0.3031064880589214,19.2,,,,JST
1880,ESP,0.3074299799944309,15.78,,,,JST
1881,ESP,0.3094795219468866,11.95,,,,JST
1882,ESP,0.3207519551332187,10.75,,,,JST
1883,ESP,0.3057220408735227,6.64,,,,JST
1884,ESP,0.2783949299706558,6.56,,,,JST
1885,ESP,0.2749790409355164,6.9,,,,JST
1886,ESP,0.2729295100726172,6.45,,,,JST
1887,ESP,0.2630234350816002,5.93,,,,JST
1888,ESP,0.2678056709192151,5.49,,,,JST
1889,ESP,0.2722463249680829,5.28,,,,JST
1890,ESP,0.2862514730788334,5.14,,,,JST
1891,ESP,0.2664393238926142,5.47,,,,JST
1892,ESP,0.2739542750224791,6.24,,,,JST
1893,ESP,0.2667809049911225,6.14,,,,JST
1894,ESP,0.2568748309073702,5.77,,,,JST
1895,ESP,0.2715631518926537,5.72,,,,JST
1896,ESP,0.2589243618819866,5.96,,,,JST
1897,ESP,0.2814692259635297,6.17,,,,JST
1898,ESP,0.3091379300852369,9.09,,,,JST
1899,ESP,0.31460334999125,6.39,,,,JST
1900,ESP,0.3303164369909255,5.29,,0.0374534018337726,0.0419478863477706,JST
1901,ESP,0.3309996208744967,5.53,0.0219212938100099,0.0367673709988594,0.0222680754959583,JST
1902,ESP,0.3234846590365821,4.78,-0.0387722477316856,0.0383725464344024,0.0438395552337169,JST
1903,ESP,0.333732326158566,4.45,0.0852419286966323,0.0354710482060909,0.0540446043014526,JST
1904,ESP,0.3398809298608811,4.59,-0.0958611592650413,0.0393602065742015,-0.0013100476935505,JST
1905,ESP,0.3415888691684836,4.34,-0.0798824056982994,0.0442525930702686,-0.057440310716629,JST
1906,ESP,0.3323659679713459,4.17,0.0072436057962477,0.0453192330896854,0.1047434359788894,JST
1907,ESP,0.3463711160492775,4.27,-0.0545091480016708,0.0481095090508461,0.0770718455314636,JST
1908,ESP,0.3368066220966839,4.16,0.0016653387574478,0.0488315671682357,0.0424919165670871,JST
1909,ESP,0.3323659679713459,4.09,0.0397191792726516,0.0459006316959857,0.0484555773437023,JST
1910,ESP,0.3354402761073622,4.18,-0.0074202502146363,0.0445692092180252,0.0242380499839782,JST
1911,ESP,0.3234846590365821,4.18,-0.0513659492135047,0.0484055653214454,0.021169040352106,JST
1912,ESP,0.3395393381012502,4.28,0.118662841618061,0.0393211208283901,0.0533804707229137,JST
1913,ESP,0.3415888691684836,4.4,0.0138699728995561,0.0394002757966518,0.0275302864611148,JST
1914,ESP,0.3405641029084553,4.22,-0.0099310958757996,0.0404210723936557,-0.0723688229918479,JST
1915,ESP,0.3730150420274868,4.47,-0.0252443980425596,0.0425890125334262,0.1292514503002166,JST
1916,ESP,0.4399664631888785,4.31,0.0636858791112899,0.039839282631874,0.0904203131794929,JST
1917,ESP,0.5127248958770176,4.31,0.1191159635782241,0.0431413352489471,0.1515799462795257,JST
1918,ESP,0.7450053309429933,4.12,0.3157175779342651,0.0339037850499153,0.1326514482498169,JST
1919,ESP,0.760718416753869,4.16,0.1021511182188987,0.0383278578519821,0.0214683096855878,JST
1920,ESP,0.7774562699094358,4.44,0.2349123954772949,0.0414667055010795,0.0640176609158515,JST
1921,ESP,0.6264739842794006,4.72,-0.3051118552684784,0.063663862645626,-0.0500439964234828,JST
1922,ESP,0.5479085497381593,4.62,-0.0461268201470375,0.066210076212883,0.2126449048519134,JST
1923,ESP,0.5520076130736815,4.53,0.0288283284753561,0.0650896057486534,0.1751825362443924,JST
1924,ESP,0.612127255047958,4.57,0.0486754104495048,0.0659363567829132,0.09121785312891,JST
1925,ESP,0.6216917492917946,4.58,0.1197633296251297,0.058090005069971,0.0510161370038986,JST
1926,ESP,0.5694286490215961,4.71,-0.0069159758277237,0.0597141534090042,0.1769116967916488,JST
1927,ESP,0.5588393898587377,4.61,-0.1009538918733596,0.0694886073470115,0.3053540885448456,JST
1928,ESP,0.5728445382693444,4.29,-0.1211601644754409,0.0816129818558693,0.2496404200792312,JST
1929,ESP,0.5888992167821371,4.34,0.0003528954403009,0.0903511121869087,0.0255135409533977,JST
1930,ESP,0.5909487492697648,4.48,0.4561940729618072,0.0595509856939315,0.0041283345781266,JST
1931,ESP,0.5947062177813288,5.53,-0.039851751178503,0.0613410286605358,-0.2420930117368698,JST
1932,ESP,0.5902655639269418,5.03,-0.1376501619815826,0.0716147497296333,-0.094573862850666,JST
1933,ESP,0.5639632297996959,4.83,-0.0088506443426013,0.0717686340212822,0.1209597215056419,JST
1934,ESP,0.5755772429347892,4.58,0.2996726632118225,0.0557883344590663,-0.0179753880947828,JST
1935,ESP,0.5885576247874126,4.19,-0.1116643846035003,0.0605347417294979,0.5931804776191711,JST
1936,ESP,0.5994884648404608,4.30755754627735,-0.0277693625539541,0.0751646161079406,-0.0659040510654449,JST
1937,ESP,0.6732716630360474,,0.2895363867282867,0.0583177357912063,0.0878115072846412,JST
1938,ESP,0.7409062573414761,,0.4061685800552368,0.0414938665926456,0.0878115072846412,JST
1939,ESP,0.8389422699892648,,-0.1154731214046478,0.0469346307218074,0.0878115072846412,JST
1940,ESP,0.9977810884319988,4.488330341113106,0.1429277062416076,0.0412889495491981,0.0978361517190933,JST
1941,ESP,1.19556104149745,4.426247648555936,0.1436333358287811,0.0362945534288883,0.3014000058174133,JST
1942,ESP,1.364305944831929,4.4145237832468815,0.0638992115855217,0.0343491435050964,0.1356231719255447,JST
1943,ESP,1.5620859101483209,4.41988950276243,0.1519818007946014,0.0299658477306365,-0.0723323598504066,JST
1944,ESP,1.6819836030175517,4.358722894192002,-0.1226040795445442,0.0343188792467117,0.0197665933519601,JST
1945,ESP,1.8298915796402755,4.349244318799609,0.2640551924705505,0.0272783152759075,0.1719476580619812,JST
1946,ESP,2.178995400217605,4.408685109666042,-0.0858151540160179,0.0302524305880069,0.3594751358032226,JST
1947,ESP,2.623060935923138,4.3763676148796495,0.1626156568527221,0.0268058460205793,0.4366329908370971,JST
1948,ESP,2.813467200714304,4.540295119182748,0.046996247023344,0.0260158907622098,-0.2906784117221832,JST
1949,ESP,2.965630400953257,4.626416840157298,-0.0438058674335479,0.0277168471366167,-0.1040574461221695,JST
1950,ESP,3.287347001653823,4.776119402985075,-0.0618063583970069,0.0301185566931962,0.0661356151103973,JST
1951,ESP,3.597263098829164,4.753416518122401,0.1334601640701294,0.030518103390932,0.1954616755247116,JST
1952,ESP,3.525839598942266,4.7511580947856045,0.0077590765431523,0.0324327647686004,0.096296302974224,JST
1953,ESP,3.5829784005844516,4.785833931562575,0.224649503827095,0.0329214744269847,0.0413147546350955,JST
1954,ESP,3.6270747013703857,4.81869654258523,0.1804903745651245,0.0296894125640392,0.2604258060455322,JST
1955,ESP,3.7730271997223017,4.844374470146542,0.0365501046180725,0.02864252589643,0.377144992351532,JST
1956,ESP,3.9941296992933193,4.866772113395791,0.0271813590079545,0.0284986421465873,0.4941006898880005,JST
1957,ESP,4.424534300072505,4.990019960079841,0.1060683876276016,0.0307437647134065,0.0921683609485626,JST
1958,ESP,5.015964901512411,5.0006250781347665,0.0399716310203075,0.0357647314667701,-0.118167832493782,JST
1959,ESP,5.382287700859441,5.0012503125781445,0.4530082046985626,0.0290662143379449,-0.0538770630955696,JST
1960,ESP,5.447371898986122,4.900159255175794,-0.2313524633646011,0.0382773801684379,0.02200866676867,JST
1961,ESP,5.5566418001133515,4.569339730408956,0.3300997912883758,0.0273905172944068,0.2837013304233551,JST
1962,ESP,5.873206501009648,4.351610095735422,0.1769192665815353,0.0286470502614974,0.2533000707626343,JST
1963,ESP,6.386227599227279,4.189797842254111,0.1337920725345611,0.0293097626417875,0.0889118835330009,JST
1964,ESP,6.832211402524457,4.23728813559322,0.1402118951082229,0.030959753319621,0.0159901604056358,JST
1965,ESP,7.641313603272756,4.184538131603724,0.1448513716459274,0.0374178849160671,0.0991163849830627,JST
1966,ESP,8.218578596649095,4.059677255658175,0.0865073502063751,0.0443140156567096,0.0783480703830719,JST
1967,ESP,8.742772499619223,3.86286817962337,0.0387361720204353,0.0563105084002018,0.0803629457950592,JST
1968,ESP,9.176652401674168,3.6032789838753265,0.0561339706182479,0.0665713250637054,0.2247946411371231,JST
1969,ESP,9.37403979739666,3.53356890459364,0.2142400294542312,0.0575900897383689,0.5244439840316772,JST
1970,ESP,9.91219980487014,3.624830086089715,0.089692011475563,0.0553757920861244,0.0952578261494636,JST
1971,ESP,10.726888004104415,3.369556060988965,0.1211879029870033,0.0523363724350929,0.0405203364789485,JST
1972,ESP,11.61233900379079,3.1680440771349865,0.1721067577600479,0.046943262219429,0.3475124835968017,JST
1973,ESP,12.940978998850644,4.786898066728948,0.1018180400133133,0.046496782451868,0.2975999414920807,JST
1974,ESP,14.878542003968924,4.421564004322625,0.1058645471930503,0.0479308143258094,0.0412130244076252,JST
1975,ESP,17.506959997412125,4.744478532293131,0.0306521169841289,0.0529218837618827,-0.1133933886885643,JST
1976,ESP,20.594397010128212,5.674551714886196,0.1096498221158981,0.0453931055963039,-0.1093562021851539,JST
1977,ESP,25.648258998217827,6.111111111111111,0.1930433213710785,0.0430136434733867,-0.2875902056694031,JST
1978,ESP,30.719811985589992,9.84724088172364,0.2529749870300293,0.0386602208018302,-0.1326136440038681,JST
1979,ESP,35.530664984532685,13.308194836048452,0.1955804377794265,0.0367072448134422,-0.0715109258890152,JST
1980,ESP,41.05751202042333,15.960833333333332,0.0673892125487327,0.0385367050766944,0.0096159568056464,JST
1981,ESP,47.0322059959101,15.811666666666667,0.0785903558135032,0.0404728204011917,0.4241999983787536,JST
1982,ESP,53.81134599319745,15.9875,0.0682861879467964,0.0424756631255149,7.298133823496755e-06,JST
1983,ESP,60.36330498761555,16.909166666666668,0.0735504701733589,0.0429565273225307,0.1242060288786888,JST
1984,ESP,67.17130799541991,16.5225,0.0935926213860511,0.0414869524538517,0.4670845866203308,JST
1985,ESP,73.09199900285351,13.3675,0.1128405258059501,0.039281565696001,0.3446483910083771,JST
1986,ESP,79.52105600491339,11.354166666666666,0.1799180805683136,0.0357954055070877,1.1325998306274414,JST
1987,ESP,83.69412400629847,12.813333333333333,0.2262890785932541,0.0307205710560083,0.479291558265686,JST
1988,ESP,87.74242799785586,11.744166666666668,0.2499616593122482,0.0261831115931272,0.1438328176736831,JST
1989,ESP,93.70222397299456,13.7025,0.2329981029033661,0.0231696516275405,0.126303568482399,JST
1990,ESP,100.00000000119084,14.6775,0.1562305390834808,0.0217325519770383,-0.117974504828453,JST
1991,ESP,105.93466000467863,12.360833333333334,0.1432258635759353,0.0207060445100069,0.0541034005582332,JST
1992,ESP,112.21102002980528,11.693333333333332,-0.0133968833833932,0.0227442421019077,-0.0808280557394027,JST
1993,ESP,117.33750003952984,10.211666666666668,-0.0040713832713663,0.0248354766517877,0.5609999895095825,JST
1994,ESP,122.87272997079562,9.995833333333334,0.0070478096604347,0.0260883029550313,-0.0920000001788139,JST
1995,ESP,128.61837995040491,11.27,0.0309445969760417,0.026681575924158,0.1570000052452087,JST
1996,ESP,133.193669939459,8.736666666666666,0.0258009899407625,0.0280849188566207,0.4289999902248382,JST
1997,ESP,135.6939412944318,6.401666666666666,0.0429664403200149,0.0286808721721172,0.4539999961853027,JST
1998,ESP,138.08577207508907,4.833333333333333,0.0487869158387184,0.0287266131490468,0.395000010728836,JST
1999,ESP,141.17325937022324,4.7275,0.0697406753897667,0.0278259832412004,0.1969999969005584,JST
2000,ESP,146.0919601372892,5.525833333333334,0.0750865489244461,0.0268628522753715,-0.1040000021457672,JST
2001,ESP,151.3359826273017,5.116666666666666,0.0947993099689483,0.0255779270082712,-0.0363768935203552,JST
2002,ESP,156.03371014859266,4.95916666666667,0.1693464666604995,0.0228017959743738,-0.2046543061733245,JST
2003,ESP,160.7764507384674,4.1241666666666665,0.1997265070676803,0.0198164600878953,0.3297805488109588,JST
2004,ESP,165.6624147283815,4.10333333333333,0.1830530762672424,0.0174369942396879,0.2308199852705001,JST
2005,ESP,171.2419891842005,3.3875,0.1456549912691116,0.0158705674111843,0.2578282356262207,JST
2006,ESP,177.2614640829975,3.785,0.1004429906606674,0.0150508284568786,0.4054714739322662,JST
2007,ESP,182.20267138435537,4.307499999999999,0.0547783747315406,0.0148925138637423,0.0916206240653991,JST
2008,ESP,189.6277816521101,4.36666666666667,0.0021031307987868,0.0154920872300863,-0.3694705069065094,JST
2009,ESP,189.08353273196053,3.97916666666667,-0.0756976604461669,0.0172760225832462,0.3560526967048645,JST
2010,ESP,192.48611150718125,4.25083333333333,-0.0264849308878183,0.0179430898278951,-0.1466962099075317,JST
2011,ESP,198.6385795631577,5.44,-0.0554687082767486,0.0191939417272806,-0.0877072066068649,JST
2012,ESP,203.49589887306388,5.845,-0.0873351320624351,0.021146323531866,0.0501080714166164,JST
2013,ESP,206.362412922423,4.56166666666666,-0.1557201296091079,0.0249999985098838,0.2988026738166809,JST
2014,ESP,206.0514135394804,2.72333333333333,0.003145863302052,0.0247502624988555,0.0908464267849922,JST
2015,ESP,205.02020505919705,1.73583333333333,0.035854909569025,0.0237784963101148,-0.0319905988872051,JST
2016,ESP,204.6048571990829,1.39333333333,0.0462261438369751,0.0227078907191753,0.0227000005543231,JST
2017,ESP,208.60692820589696,1.555,0.0619623661041259,0.0214711930602788,0.0971999987959861,JST
2018,ESP,212.10157916685728,1.4225,0.0672801733016967,0.0203791987150907,-0.1088000014424324,JST
2019,ESP,213.5849643815506,0.659166667,0.0517346858978271,0.0196843650192022,0.1623000055551529,JST
2020,ESP,212.8954460127897,0.375,0.0223709344863891,0.0194714423269033,-0.0980999991297721,JST
2021,ESP,219.29963878711084,0.3474166666666667,0.03295162906231752,0.018962765622591443,0.3192769432512965,extension
2022,ESP,237.55099062861788,2.1805,0.07481091110205718,0.017880047590475057,-0.12305995046665219,extension
2023,ESP,245.62836626586483,3.4761666666666664,0.041162391660243935,0.017525803874412386,0.20621503077457937,extension
2024,ESP,252.67110025522092,3.1485833333333333,0.07917226221595874,0.016596239457949673,0.24166793616665183,extension
2025,ESP,259.47480812130186,3.2175833333333332,0.1257843280299471,0.015097758835192838,0.09075463061088596,extension
'''
annual = pd.read_csv(io.StringIO(DATA_CSV))
annual.tail(8)

In [ ]:
# MODEL: presný výpočet oboch stratégií (src/model.py)
"""Kúpa s hypotékou vs. prenájom a investovanie: jeden výpočtový model pre obe úrovne.

Pravidlo férovosti: obe domácnosti míňajú v každom období rovnako. Kto má v danom období
lacnejšie bývanie, rozdiel investuje do rovnakého akciového indexu. Pre každý horizont h obe
domácnosti „speňažíme“ a porovnáme čistý majetok:

    kupujúci = predajná cena - náklady na predaj - zostatok hypotéky + jeho portfólio (po zdanení)
    nájomník = portfólio (po zdanení)

    výhoda = kupujúci - nájomník      (> 0  ->  kúpa vyhráva)

Všetko je vyjadrené na 1,0 kúpnej ceny, takže výsledok nezávisí od výšky ceny.
"""
from dataclasses import dataclass, replace
import numpy as np
import pandas as pd


@dataclass(frozen=True)
class Params:
    ltv: float = 0.80            # pomer úveru k cene (LTV)
    buy_cost: float = 0.10       # dane a poplatky pri kúpe (podiel ceny)
    sell_cost: float = 0.04      # makléri + miestne dane pri predaji (podiel ceny)
    upkeep: float = 0.012        # údržba + daň z nehnuteľnosti + poistenie, ročne, z aktuálnej hodnoty
    gross_yield: float = 0.05    # nájom v 1. roku / cena
    term_years: int = 30         # zadanie projektu: hypotéka štandardne na 30 rokov
    spread: float = 0.0          # variabilné úvery: Euribor + marža (určená pri čerpaní)
    invest_tax: float = 0.21     # daň zo zisku portfólia pri speňažení
    invest_tax_free_after: int = 0  # po koľkých obdobiach je zisk z portfólia bez dane (0 = nikdy)
    home_gain_tax: float = 0.0   # daň zo zisku z predaja bytu (v Španielsku sa predpokladá úľava na hlavné bývanie)
    home_gain_tax_free_after: int = 0  # po koľkých obdobiach je zisk z bytu oslobodený od dane (SK: 5 rokov)


SPAIN = Params()
SLOVAKIA = Params(
    buy_cost=0.006,          # poplatok katastru + právnik; bez dane z prevodu od 2005
    sell_cost=0.03,          # provízia makléra
    upkeep=0.0105,           # údržba ≈ 1 % + veľmi nízka daň z nehnuteľnosti
    gross_yield=0.045,
    term_years=30,
    invest_tax=0.0,          # zisky z ETF bez dane po viac ako 1 roku ...
    home_gain_tax=0.19,      # ... zisk z bytu zdanený 19 % pri predaji do 5 rokov
)


def annuity(rate_per_period: float, n: int) -> float:
    """Splátka na 1,0 úveru."""
    if abs(rate_per_period) < 1e-12:
        return 1.0 / n
    return rate_per_period / (1 - (1 + rate_per_period) ** -n)


def simulate(price_g, rent_g, eq_r, mort_rate_path, p: Params, ppy: int, horizon: int, parts: bool = False):
    """Simuluje jeden dátum nákupu na `horizon` období.

    price_g, rent_g, eq_r : rast cien / nájmov / výnos investície za obdobie (desatinné čísla)
    mort_rate_path        : ROČNÁ sadzba hypotéky v každom období (pri fixnej konštantná)
    Vracia pole adv[1..horizon] = majetok kupujúceho - nájomníka (na 1,0 ceny).
    S parts=True vráti (kupujúci, nájomník, adv): čistý majetok každého scenára po speňažení v každom období.
    """
    n_total = p.term_years * ppy
    price = 1.0
    rent = p.gross_yield / ppy
    loan = p.ltv * 1.0
    bal = loan
    pay_n_left = n_total
    # počiatočná hotovosť: akontácia + náklady na kúpu idú do portfólia nájomníka
    renter = (1 - p.ltv) + p.buy_cost
    renter_contrib = renter
    buyer_port = 0.0
    buyer_contrib = 0.0
    adv = np.zeros(horizon)
    buyer_arr = np.zeros(horizon)
    renter_arr = np.zeros(horizon)
    for t in range(horizon):
        r_per = mort_rate_path[t] / ppy
        pay = annuity(r_per, pay_n_left) * bal if bal > 1e-12 else 0.0
        interest = bal * r_per
        bal = max(bal - (pay - interest), 0.0)
        pay_n_left = max(pay_n_left - 1, 1)
        buy_out = pay + p.upkeep / ppy * price
        rent_out = rent
        diff = buy_out - rent_out
        # najprv rast existujúcich portfólií, potom nový vklad na konci obdobia
        renter *= 1 + eq_r[t]
        buyer_port *= 1 + eq_r[t]
        if diff > 0:               # kúpa stojí viac hotovosti -> nájomník investuje rozdiel
            renter += diff
            renter_contrib += diff
        else:                      # prenájom stojí viac hotovosti -> rozdiel investuje kupujúci
            buyer_port += -diff
            buyer_contrib += -diff
        price *= 1 + price_g[t]
        rent *= 1 + rent_g[t]
        periods = t + 1
        # hodnoty po speňažení
        tax_free = p.invest_tax_free_after and periods > p.invest_tax_free_after
        tr = 0.0 if tax_free else p.invest_tax
        r_net = renter - tr * max(renter - renter_contrib, 0)
        b_net = buyer_port - tr * max(buyer_port - buyer_contrib, 0)
        home_tax_free = p.home_gain_tax_free_after and periods > p.home_gain_tax_free_after
        hg = 0.0 if home_tax_free else p.home_gain_tax
        gain = max(price - 1.0, 0)
        home_net = price * (1 - p.sell_cost) - hg * gain - bal
        adv[t] = home_net + b_net - r_net
        buyer_arr[t], renter_arr[t] = home_net + b_net, r_net
    return (buyer_arr, renter_arr, adv) if parts else adv


def slovakia_flags(p: Params, ppy: int) -> Params:
    """SK: zisky z ETF bez dane po 1 roku; zisk z bytu bez dane po 5 rokoch."""
    return replace(p, invest_tax_free_after=ppy, home_gain_tax_free_after=5 * ppy)


# ---------------------------------------------------------------- úroveň 1 (mesačne)
def tier1_grid(monthly: pd.DataFrame, city: str, rate_type: str, p: Params | None = None,
               max_years: int = 20, start_from: str | None = None, es_prices: str = "hp_ma",
               sk_prices: str = "hp_ba"):
    """Spätný test pre každý mesiac nákupu. Riadky = mesiac nákupu, stĺpce = horizont v rokoch."""
    if city == "ES":
        # es_prices: "hp_ma" = MIVAU valor tasado pre obec Málaga (od 2005-03), "hp_es" = celoštátny index BIS
        hp, hicp_rent, mir = monthly[es_prices], monthly["hicp_rent_es"], monthly["mir_es"]
        p = p or SPAIN
        start_from = start_from or ("2005-03" if es_prices == "hp_ma" else "2005-01")
    else:
        # sk_prices: "hp_ba" = NBS ponukové ceny, Bratislavský kraj; "hp_sk" = celoštátny index BIS
        hp, hicp_rent, mir = monthly[sk_prices], monthly["hicp_rent_sk"], monthly["mir_sk"]
        p = slovakia_flags(p or SLOVAKIA, 12)
        start_from = start_from or "2008-01"
    eq = monthly["msci_eur_tr"]
    ph = hp.pct_change().shift(-1)           # rast z t na t+1
    rg = hicp_rent.pct_change().shift(-1)
    er = eq.pct_change().shift(-1)
    idx = monthly.index
    out = {}
    for i, m in enumerate(idx):
        if m < pd.Period(start_from, "M"):
            continue
        H = min(max_years * 12, len(idx) - i - 1)
        if H < 12:
            break
        rate0 = mir.iloc[i] / 100
        if rate_type == "fixed":
            path = np.full(H, rate0)
        else:
            spread = rate0 - monthly["euribor12m"].iloc[i] / 100
            path = np.clip(monthly["euribor12m"].iloc[i:i + H].values / 100 + spread, 0.005, None)
        adv = simulate(ph.iloc[i:i + H].values, rg.iloc[i:i + H].values, er.iloc[i:i + H].values,
                       path, p, 12, H)
        out[m] = {y: adv[y * 12 - 1] for y in range(1, max_years + 1) if y * 12 <= H}
    return pd.DataFrame(out).T


# ---------------------------------------------------------------- úroveň 2 (ročne, JST)
def tier2_run(jst: pd.DataFrame, year: int, horizon: int, iso: str = "ESP", p: Params | None = None,
              rate_spread: float = 0.015, parts: bool = False):
    """Jeden ročný nákup v roku `year` na `horizon` rokov; skutočné dáta JST (a predĺženie 2021–2025).

    Vracia (adv, cpi_pomer) alebo s parts=True (kupujúci, nájomník, adv, cpi_pomer); pomer CPI[k] =
    CPI(year+k) / CPI(year) slúži na prepočet na peniaze roku nákupu. Všetko na 1,0 ceny bytu."""
    d = jst[jst.iso == iso].set_index("year").sort_index()
    p = p or SPAIN
    cols = pd.concat([d["housing_capgain"], d["housing_rent_yd"], (d["ltrate"] / 100 + rate_spread).clip(lower=0.01), d["eq_tr"]],
                     axis=1, keys=["g", "ry", "rate", "eq"]).dropna()
    years = list(cols.index)
    i = years.index(year)
    H = horizon
    P = np.r_[1.0, np.cumprod(1 + cols["g"].values[i + 1:i + H + 1])]   # cena na začiatku roka k
    rent_path = cols["ry"].values[i:i + H + 1] * P                       # nájom_k = výnos_k * cena_k
    rg = rent_path[1:] / rent_path[:-1] - 1
    pp = replace(p, gross_yield=cols["ry"].values[i])
    out = simulate(cols["g"].values[i + 1:i + H + 1], rg, cols["eq"].values[i + 1:i + H + 1],
                   np.full(H, cols["rate"].values[i]), pp, 1, H, parts=parts)
    cpi = d["cpi"].reindex(years).values
    deflator = cpi[i + 1:i + H + 1] / cpi[i]
    return (*out, deflator) if parts else (out, deflator)


def tier2_grid(jst: pd.DataFrame, iso: str = "ESP", p: Params | None = None,
               start_from: int = 1950, max_years: int = 30, rate_spread: float = 0.015):
    """Ročný spätný test od `start_from` na národných dátach (reálne hodnoty, v peniazoch roku nákupu)."""
    d = jst[jst.iso == iso].set_index("year").sort_index()
    cols = d[["housing_capgain", "housing_rent_yd", "ltrate", "eq_tr"]].dropna()
    years = cols.index.values
    out = {}
    for i, y in enumerate(years):
        if y < start_from:
            continue
        H = min(max_years, len(years) - i - 1)
        if H < 5:
            break
        adv, defl = tier2_run(jst, int(y), H, iso, p, rate_spread)
        out[int(y)] = {h: adv[h - 1] / defl[h - 1] for h in range(1, H + 1)}
    return pd.DataFrame(out).T


def break_even(row: pd.Series):
    """Prvý horizont, od ktorého kúpa ostáva vpredu (None, ak nikdy)."""
    ok = row.dropna()
    wins = ok > 0
    for h in ok.index:
        if wins.loc[h:].all():
            return h
    return None


# ---------------------------------------------------------------- vektorizovaný model (pre sieť)
def simulate_batch(g, rg, eq, rate, ltv, buy_cost, sell_cost, upkeep, gross_yield, term_years,
                   invest_tax, inv_free_after, home_tax, home_free_after, years):
    """Rovnaká ekonomika ako `simulate`, mesačne, konštantné ročné sadzby, pre N scenárov naraz.

    Všetky argumenty sú polia tvaru (N,). `years` je horizont v rokoch; vracia výhodu
    (kupujúci - nájomník) na 1,0 ceny v danom horizonte.
    """
    N = len(g)
    gm = (1 + g) ** (1 / 12) - 1
    rgm = (1 + rg) ** (1 / 12) - 1
    em = (1 + eq) ** (1 / 12) - 1
    r = rate / 12
    n_left = (term_years * 12).astype(float)
    price = np.ones(N)
    rent = gross_yield / 12
    bal = ltv.copy()
    renter = (1 - ltv) + buy_cost
    renter_c = renter.copy()
    bport = np.zeros(N)
    bport_c = np.zeros(N)
    H = (years * 12).astype(int)
    result = np.zeros(N)
    for t in range(int(H.max())):
        # anuitná splátka zo zostatku
        pay = np.where(np.abs(r) < 1e-12, 1.0 / n_left, r / (1 - (1 + r) ** (-n_left))) * bal
        interest = bal * r
        bal = np.maximum(bal - (pay - interest), 0)
        n_left = np.maximum(n_left - 1, 1)
        diff = pay + upkeep / 12 * price - rent
        renter = renter * (1 + em)
        bport = bport * (1 + em)
        pos = diff > 0
        renter = renter + np.where(pos, diff, 0)
        renter_c = renter_c + np.where(pos, diff, 0)
        bport = bport + np.where(pos, 0, -diff)
        bport_c = bport_c + np.where(pos, 0, -diff)
        price = price * (1 + gm)
        rent = rent * (1 + rgm)
        done = (H == t + 1)
        if done.any():
            per = t + 1
            tr = np.where((inv_free_after > 0) & (per > inv_free_after), 0.0, invest_tax)
            hg = np.where((home_free_after > 0) & (per > home_free_after), 0.0, home_tax)
            r_net = renter - tr * np.maximum(renter - renter_c, 0)
            b_net = bport - tr * np.maximum(bport - bport_c, 0)
            home_net = price * (1 - sell_cost) - hg * np.maximum(price - 1, 0) - bal
            result = np.where(done, home_net + b_net - r_net, result)
    return result

In [ ]:
# VÁHY NEURÓNOVEJ SIETE z webu (natrénované na simulovaných scenároch, src/train_nn.py)
NN_JSON = '{"features":["city_SK","years","ltv","gross_yield","buy_cost","house_growth","rent_growth","equity_return","mortgage_rate"],"gapScale":3.0,"ranges":{"gross_yield":[0.01,0.07],"house_growth":[-0.05,0.2],"rent_growth":[0.0,0.2],"equity_return":[0.0,0.2],"mortgage_rate":[0.01,0.2]},"mu":[0.50105,16.015158333333332,0.7000276324587796,0.0399363611727126,0.05187149686434251,0.07503590119694524,0.060521491647855315,0.09988789724112451,0.10504838019524872],"sd":[0.49999889849881113,8.36164429054787,0.11557947422517008,0.017313696494171294,0.04336410551316876,0.0720527487482598,0.04931764247996743,0.05769101062924555,0.054787816619567775],"layers":[{"W":[[0.184626,0.823262,0.344921,-0.605762,0.360893,-0.632719,0.111965,0.713124,-1.082094,0.330176,0.482123,-0.072088,0.535774,0.286103,0.289857,-0.104847,-0.048071,0.534815,-0.079526,0.725888,0.051594,-0.307587,0.532181,0.610654,-0.128251,0.859975,-0.150751,-0.193402,-0.844328,-0.164625,-0.610111,0.676348,-1.051952,-0.374148,0.311528,-0.254867,-0.792303,0.379403,0.330975,-0.347842,0.398943,-0.270325,0.271029,0.634418,0.689786,-0.048718,0.593573,0.276338,-0.379125,-0.159919,-0.08956,0.610269,0.606582,0.123797,-0.4774,0.101103,-0.114679,-0.138586,-0.109225,0.521919,0.462215,0.689779,-0.62473,-0.172895],[0.433673,-0.056734,-0.017328,0.029783,0.31489,0.429483,-0.46083,0.004975,-0.478388,0.295625,0.033522,-0.476562,0.871812,0.09874,-0.881919,-0.032163,0.893277,-0.24954,-0.728987,0.500015,1.071653,0.588544,0.790558,0.422836,-2.053179,-0.879193,0.111921,-0.945765,0.402734,0.242786,-0.297663,-0.70576,0.563723,0.14229,0.776481,0.391683,-0.193636,0.438198,0.120567,0.013007,0.829677,0.509098,0.824453,0.597073,-0.390233,-0.682863,0.280387,0.798825,0.451837,0.3151,-0.262596,-0.332476,-0.337051,-0.829274,-0.072054,0.301379,-0.896953,0.136646,0.548372,0.130408,-0.081231,0.146721,-0.58248,-0.657194],[0.231371,-0.46834,-0.185411,-0.894562,-1.11979,-0.490026,0.184383,-0.00968,0.062255,-0.854883,-0.194903,-0.248086,-0.243621,0.168946,0.029632,0.356593,0.050491,0.337013,-0.520527,-0.109553,0.13221,-0.077716,0.027245,0.646852,0.107727,-0.365857,0.009765,0.735177,-0.08188,-0.943721,0.251816,-0.012644,-0.026536,0.091543,-0.124838,0.12588,0.221646,0.277425,-0.060999,0.689674,0.172286,-0.037015,0.424324,0.247112,0.569376,0.210234,0.608865,0.313588,-0.293426,0.224595,-0.116707,0.46754,0.028986,0.038055,0.28299,0.055704,0.130623,-0.883291,-0.097492,0.890507,0.106284,0.057858,-0.206588,-0.984095],[0.075326,-0.030257,0.398712,-1.103391,-0.645306,-0.543543,-0.238936,-0.086131,-0.293746,-0.012906,-0.130826,0.026204,-0.083282,0.731963,-0.288325,0.179857,-0.403433,-0.459611,0.21139,-0.178061,0.321922,0.336088,0.015584,0.275523,0.07375,0.289195,-0.027571,0.723819,-0.101883,0.527709,-0.067416,0.733543,-0.129396,0.408025,0.190445,1.148075,0.186583,0.14799,-0.452007,-0.054164,0.520929,-0.298585,-0.757857,-0.543563,-0.168722,-0.004697,-0.18114,0.241547,0.274347,-1.055529,0.242801,0.066705,0.586786,-0.299244,0.432174,0.073721,-0.306952,0.257858,1.104263,0.171379,-0.595247,-0.489841,-0.365413,-0.224664],[0.336098,1.073072,0.411446,-0.271143,0.035431,0.216952,0.372561,0.180971,0.047096,0.204548,0.999409,-0.968922,-0.073576,0.159849,0.108337,0.199349,-0.116924,1.196972,0.64206,0.073753,-0.115433,-0.167032,0.082067,-0.290274,0.092865,-0.04945,0.213218,-0.424255,-0.019758,-0.604372,0.241276,0.759281,-0.135062,0.059247,-0.010329,-0.205459,0.268178,0.67138,0.655473,-0.220561,0.249005,0.247093,-0.016788,-0.42737,-0.032075,-0.368404,-0.152665,0.296035,-0.057232,-0.1818,-0.151453,-0.468547,0.461933,0.212918,-0.80933,0.58531,0.115563,0.310595,-0.113319,-0.260722,0.25999,-0.233707,-0.344182,0.08392],[0.594064,-0.775707,1.291167,-0.13203,0.104429,-0.805409,-1.269731,-0.118251,-0.617708,0.479313,0.138068,-0.734298,-0.2545,0.637568,0.672131,-0.849454,-1.139213,0.338333,0.338349,-0.490138,-0.01497,0.944625,1.402683,2.037946,-0.058968,0.437638,1.195142,-0.398165,0.516212,0.050413,0.213004,-0.423571,-0.587162,0.293366,1.604892,0.302495,0.169146,-0.684809,0.473434,0.114013,-0.849069,-0.972468,0.02192,-0.297661,-0.724053,-0.563435,0.830464,0.378845,-0.850534,0.319473,-0.165377,0.13067,-0.540588,-1.097638,0.188297,0.578583,0.696927,0.010791,-0.506336,-0.68796,-1.241738,-0.395439,-1.353884,-0.085736],[-1.1914,-0.450337,-0.6133,-0.329855,-0.499794,-0.043392,-0.524108,0.18887,0.17178,0.38628,-0.614452,-0.267258,-0.354694,-0.365049,-0.263384,-1.194717,-0.212907,-0.466755,0.628498,-0.023938,0.510449,0.584954,-0.131872,-0.047854,-0.134355,0.099019,-0.43459,0.828872,-0.168571,0.716879,-0.219563,0.85898,-0.166094,-0.640622,0.31256,1.203017,-0.399462,-0.21961,-0.16105,-1.44847,1.324351,0.028171,-0.091306,0.177596,-0.457376,0.692666,-0.67951,0.743123,-0.296774,0.275747,0.624156,-0.620363,-0.003653,0.225387,0.877333,0.51754,0.19906,-0.004647,0.101782,0.033105,0.594219,-0.743599,-0.00137,0.155454],[0.497461,0.332871,0.509425,0.029885,0.463765,1.03769,0.569379,-0.216559,-0.415675,-0.538158,-0.228495,-0.039873,1.009731,-1.085177,0.547225,0.34321,1.021086,-0.333481,-0.704285,0.461236,0.399611,-0.380415,-0.401905,-0.511344,-0.019242,-0.191763,-0.335727,-0.250465,-0.341336,-0.261112,0.374218,0.361772,0.789919,0.294225,-1.005673,-0.636397,-0.306184,0.249323,0.698945,-0.655756,-0.546783,-0.058276,0.131691,0.633672,0.887458,0.031358,-0.263174,-0.055411,0.609194,-0.795478,-0.95314,-0.232873,-0.052378,-0.708939,-0.466912,-0.580525,-0.466487,0.592644,0.214124,0.463921,0.734807,-0.043077,0.31227,0.312621],[0.250284,0.179803,-0.27654,-0.172609,-0.131422,0.061259,0.854525,0.041096,-0.153917,0.193585,0.067612,-0.012079,-0.072835,-0.652196,-0.270559,0.253561,0.70028,-0.06439,-0.262467,0.247121,-0.511596,-0.266764,0.536434,-0.631782,-0.178872,0.175825,-0.181262,-1.29439,0.281472,0.740567,-0.471739,-0.270901,0.31659,-0.879761,-0.25346,-0.469324,0.396307,1.025805,0.10996,0.385231,-0.402451,-0.200455,0.028102,-0.049542,-0.386591,0.239393,-0.320227,-1.147646,-0.061108,-0.419595,-0.482507,0.40777,0.654205,0.416562,-0.178415,0.704543,0.414895,-0.389058,0.411091,0.662793,0.309925,0.216101,0.845834,0.099954]],"b":[-0.035837,-0.224075,-0.627303,0.167772,-0.468216,-0.025876,0.154565,0.009762,-0.213152,0.078163,-0.201695,0.211232,-0.237229,-0.124062,-0.375921,0.079231,0.107464,-0.310111,-0.047487,0.219676,-0.764854,-0.050235,-0.569062,0.079432,-1.099103,-0.420465,0.041729,-0.189306,0.211519,-0.260009,0.093036,-0.387698,-0.068889,0.03627,-0.439208,-0.406593,0.114139,0.139203,-0.010273,-0.330897,-0.71201,0.359764,-0.108814,0.076882,-0.44466,-0.345326,-0.317679,-0.11498,-0.387824,-0.254167,0.073157,-0.146803,-0.341529,-0.003807,0.262357,-0.520855,-1.176195,0.060749,-0.451196,0.081423,-0.005488,-0.115025,-0.056567,-0.211783]},{"W":[[0.094595,0.017584,0.105074,-0.171535,0.114438,0.181711,-0.119534,0.264676,0.472652,0.438843,-0.05151,-0.478957,-0.095477,-0.012471,-0.321893,0.02446,-0.329795,-0.370291,0.392373,0.039216,0.151677,-0.546858,0.159693,0.288952,0.098276,-0.266715,0.277055,0.223607,-0.450365,0.110889,-0.722178,0.101112,-0.444256,0.349996,0.327973,0.257337,-0.661502,-0.615123,0.639365,0.102034,-0.2426,-0.274763,0.018753,0.371706,-0.076869,0.128119,-0.255084,-0.185393,-0.091265,-0.011017,0.015746,0.023445,-0.904702,0.359424,-0.696397,-0.07708,-0.230458,0.306912,-0.148027,-0.200562,0.394494,0.259956,0.108128,0.087256],[-0.03637,-0.272771,-0.012798,0.251423,0.441112,0.050343,0.02948,-0.633465,-0.521726,-0.428137,0.14529,-0.026586,0.368067,0.191828,0.078605,0.188438,0.144523,-0.463657,0.4354,-0.081659,-0.187844,-0.121034,-0.316119,-0.497699,-0.14341,0.334779,-0.362581,0.224638,0.131048,0.273264,-0.315541,0.023171,0.83839,-1.199384,-0.054087,0.32624,0.223576,0.13533,-1.488337,0.50774,0.175387,0.31293,0.169592,0.323494,0.057892,0.475073,0.105286,-0.161944,-0.099473,-0.074432,-0.079853,0.199773,-2.169,0.34957,-0.136265,-0.009513,-0.422002,-0.010874,-0.044623,-0.063994,-0.596792,-0.173784,-0.281065,-0.185386],[0.118,0.068157,-0.150512,-0.23037,0.740056,0.36894,0.020849,0.30028,0.132982,-0.723754,0.255333,-0.084762,-1.031572,0.231804,0.213012,0.138658,0.125385,-0.104216,-0.093626,-0.07159,0.158006,0.316321,-0.471158,0.344283,0.106079,-0.583703,0.185692,-0.061021,0.094495,0.218869,0.22216,0.160734,-0.012725,-0.023358,0.128924,-0.265392,0.235716,-0.053229,-0.000967,0.419121,0.295903,0.307664,-0.030914,0.24027,0.312571,0.066632,1.418759,-0.946888,0.020208,0.055223,-0.040775,0.202684,0.301379,-0.537904,0.531548,-0.071861,0.003771,0.301517,0.052057,-0.002428,-0.146661,-0.224686,0.189276,-0.31083],[-0.095664,0.058095,-0.012456,0.299755,0.039994,-0.076391,0.155499,-0.103109,0.477106,-0.073321,-0.048742,0.103585,0.20516,-0.43915,0.083557,0.069097,0.005114,0.304041,-0.819153,-0.17911,0.190634,-0.122865,0.169347,-0.317725,0.316571,0.322096,-0.028731,0.103389,0.039965,-0.761242,0.351433,-0.095865,0.146534,0.135245,-0.085986,0.344126,-0.166463,-0.03568,-0.181127,0.007416,-0.196744,0.25873,-0.126055,-0.101835,-0.569352,0.28176,-0.381365,-0.412045,0.002447,0.015945,-0.047826,-0.262251,-0.99363,-0.200771,0.093066,0.014612,-0.044215,-0.073122,0.013057,-0.03225,-0.291408,0.311532,0.238779,0.060424],[0.112983,-0.043431,0.100604,-0.02709,0.164208,0.315481,-0.004617,0.156944,0.119141,-0.011797,0.234789,-0.134665,0.230352,0.070057,-0.035399,0.124656,0.19375,0.003622,0.16502,-0.131179,0.01029,-0.107618,0.025923,0.377031,-0.176438,0.393729,0.309202,0.521309,-0.115276,-0.433112,-0.476993,0.065694,0.146967,-0.29599,0.191337,0.015543,0.042832,-0.316187,-0.364604,0.574059,0.407644,0.179167,0.242981,0.702679,0.102038,0.094349,0.056631,-0.008677,-0.093998,-0.290935,-0.026556,0.267477,0.050943,-1.020303,0.314021,-0.173012,0.328522,0.255536,-0.135414,-0.316715,0.368147,0.091846,-1.143283,0.113153],[0.178999,-0.085039,-0.382598,-0.216517,-0.471158,0.436645,-0.297041,0.052026,0.195595,-0.056246,0.596245,-0.493334,0.23862,0.179813,-0.008928,0.087723,-0.103603,-0.665785,-0.379584,0.335336,0.269401,0.033021,0.167816,-0.484478,-0.441165,0.024554,0.018312,0.108801,-0.104598,-0.230925,-0.521625,0.312662,0.142075,-0.369253,0.476166,-0.067644,0.365287,0.075215,-0.404725,-1.266359,0.024113,0.1191,0.220516,-0.222565,0.170816,-0.034394,-0.046106,-0.146754,0.260772,0.148709,0.188499,-0.297382,0.085868,-0.00285,-0.404182,0.138334,-0.363033,0.439116,-0.08057,-0.139261,-0.051773,0.174349,0.114156,0.098695],[-0.033663,-0.18639,-0.403319,-0.886129,0.612047,-0.287789,0.262801,0.192795,-0.237371,0.521657,0.455719,0.249336,0.038421,-0.077376,0.32091,-0.238597,0.026653,0.457057,-0.203319,0.017681,0.219959,0.185737,0.020421,0.149823,0.163321,-0.693984,-0.557513,-0.262144,0.06481,0.17908,-0.07553,-0.023398,0.136607,-1.013849,-0.153722,-0.383282,0.456364,-0.360594,0.01085,-0.211844,0.201331,0.306899,-0.141592,-0.019523,0.090075,0.755879,0.30969,0.811153,-0.182997,0.360017,0.003574,0.365768,-0.136056,-0.038795,0.173158,0.03493,0.146274,-0.289734,0.244143,-0.356605,0.151876,0.343491,-0.303975,-0.262253],[-0.251756,0.005784,-0.11192,-0.09173,-0.247392,-0.10141,0.114212,0.217649,-0.268946,0.248354,-0.227889,0.091988,-0.338784,-0.306505,-0.417609,-0.208569,0.165504,0.190026,-0.098272,0.075795,-0.178854,-0.896007,-0.377679,-0.476923,0.247354,0.062704,0.074887,-0.286775,0.151556,-0.076734,-0.163721,0.632421,-0.282577,-0.509399,-0.11526,0.09146,0.540061,-1.177503,0.121474,-0.301186,-0.292655,0.051535,0.262087,-0.207434,0.325594,-0.078506,-0.000749,0.172831,0.132726,-0.154784,0.162036,0.0537,-0.275617,-0.381934,0.044405,0.255546,0.331623,-0.487633,-0.276734,0.109154,-0.1388,0.018065,-0.425498,-0.212089],[0.480535,0.033749,-0.36198,0.077484,-0.062592,-0.083583,-0.42198,0.040086,0.102731,-0.018843,-0.251898,-0.257491,0.117771,0.114916,0.109381,-0.15659,0.418614,-0.442874,-0.303733,0.09224,-0.23586,0.066253,-0.816557,0.202628,-0.279436,0.187258,-1.287504,0.026284,0.062322,-0.106358,0.478461,0.204623,-0.321284,0.209126,0.26985,0.421757,0.147991,0.271204,0.088088,0.344803,0.068195,0.096803,0.145337,-0.496518,0.041262,-0.341346,-0.160701,-0.212374,-0.175664,0.026542,0.26747,-0.863202,-0.542787,-0.116288,-0.147126,0.514726,0.36054,-0.197942,0.36981,0.146776,-0.014288,0.140796,0.102438,0.089867],[-0.022316,-0.084764,0.196409,0.3282,-0.152438,-0.291107,-0.206102,0.479188,0.412235,0.229859,-0.518931,0.336377,-0.988563,0.056513,0.068665,0.061838,-0.031992,0.162794,-0.466287,0.404949,0.006043,0.39852,-0.256256,-1.551941,0.16438,0.239226,-0.2631,-0.38538,0.233235,-0.219344,-0.441508,-0.162433,0.525261,-0.219018,-0.41778,0.291758,0.256371,0.14145,0.057751,0.104692,-0.156583,-0.036697,-0.022422,-0.164914,-0.310295,0.326114,0.08785,-0.35044,-0.098753,-0.480704,-0.007342,0.073037,0.100113,-0.033374,0.322147,-0.057686,0.274735,-0.25565,0.235273,0.259136,0.463077,0.119048,-1.353212,0.37065],[-0.005877,0.30012,0.032478,0.081352,-0.154181,0.342471,0.111487,0.002584,-0.01062,0.085711,0.075938,0.183945,-0.142814,-0.122615,-0.06937,-0.069119,0.227928,-0.249252,-0.580506,-0.131839,0.017345,0.489359,-0.360969,-1.060771,0.132302,0.320575,-0.148787,-0.383926,0.153302,0.601738,0.168916,-0.009906,0.267454,0.493423,-0.257597,0.549363,0.159307,0.086409,-0.334278,0.2767,-0.066326,0.125746,-0.227005,0.172113,0.159009,0.247122,0.075402,-0.218992,-0.229402,-0.118666,0.009294,0.046417,-1.642821,-0.052985,-0.581465,0.012147,-0.078789,0.275493,0.000428,-0.09045,-0.235071,0.610262,0.050767,-0.65175],[-0.26005,0.262589,-0.020091,-0.116278,-0.493365,-0.315249,0.368173,-0.293926,-0.134176,0.25559,0.400602,0.37296,-0.110416,-0.130993,-0.308855,0.239152,0.325991,0.609458,0.331278,0.277449,-0.198716,-0.622613,-0.153787,0.201237,-0.168862,0.012746,0.018693,-0.031217,0.049305,-0.145603,0.212949,-0.028889,-0.263868,0.231748,-0.369846,0.173518,-0.214267,-0.004775,-0.224636,-0.258527,-0.067926,0.351981,-0.07908,-0.106073,0.0359,-0.211429,-0.014513,-0.559628,0.05664,-0.226928,0.195978,0.234736,0.454071,0.012555,-0.119085,-0.083546,0.109117,-0.330398,-0.124695,0.034164,-0.179959,-0.38629,-0.118185,0.190299],[0.221994,0.150789,-0.481205,-0.619911,-0.220631,0.392347,-0.009815,0.266776,0.045809,-0.199012,0.16731,-0.571558,0.049176,0.222432,-0.038941,-0.256978,-0.313247,-0.396094,0.129916,0.028535,0.102265,-0.310783,0.251467,0.721695,-0.004893,-0.142248,0.538869,0.384783,-0.295355,-0.275767,0.045342,0.223767,0.128896,0.137505,0.502062,-0.007849,-0.459988,-0.266554,-0.028417,0.12419,0.214701,-0.008408,0.183758,0.008508,-0.114163,0.079803,0.161502,0.332289,0.182269,-0.059831,-0.005571,-0.027566,-0.679709,0.310291,0.480675,0.037625,-0.359123,0.382931,-0.146619,-0.066873,-0.345439,0.182011,-1.036769,0.50502],[0.177818,0.422563,-0.038742,0.642239,0.421512,-0.292311,0.119327,-0.670272,0.552773,-0.397824,-0.019302,0.210581,-0.06269,-0.004555,0.141732,0.331483,-0.682938,0.447645,0.012982,-0.208424,-0.052917,-0.184085,0.163666,0.093608,-0.11181,0.078172,0.000499,0.658988,0.236031,0.155292,0.435913,0.10513,-3.636455,0.054451,-0.249569,0.30652,-0.062738,0.03317,0.31816,0.332527,0.108038,-0.18662,-0.243517,-0.1973,-0.401362,-1.301843,0.322029,0.354553,-0.009089,0.172302,0.07083,0.134639,1.235461,-0.667133,0.068754,-0.161839,-0.155042,-0.405628,0.027184,0.067642,-0.20451,0.22441,-0.352922,-0.444816],[-0.021028,0.260891,-0.020927,-0.233788,0.444209,-0.534815,-0.172305,-0.173512,0.050741,-0.551782,0.039297,-0.012939,0.113666,-0.478651,0.133015,0.378517,-0.222193,0.138972,0.108779,0.028171,0.028262,-0.086247,-0.290199,-0.014718,-0.143167,0.370899,-0.113983,0.060999,0.052132,0.216692,0.336829,0.139957,0.11303,0.05276,0.069828,0.078485,-0.343345,0.148448,0.413757,0.253619,-0.384119,0.137399,-0.339044,-0.526832,0.03156,0.437911,-1.24927,0.04474,0.150247,0.244648,0.108974,-0.057893,0.021028,0.267705,0.22111,-0.060778,0.006474,-0.460139,0.189136,0.165902,-0.998715,0.452433,-0.293598,-0.219127],[0.039371,-0.5279,-0.148264,-1.566795,0.288223,0.018655,0.093765,-0.167753,-0.671514,-0.058423,-0.088032,0.098843,0.292498,0.020778,0.109372,0.270494,0.25431,0.150476,-0.17297,-0.05429,0.07964,0.112859,0.018361,0.351474,-0.197549,-0.066251,-0.353758,-0.317068,0.172277,0.102874,-0.139997,0.033705,-0.450321,-0.904025,-0.033026,-0.070999,0.013253,0.296989,0.059688,-0.008334,0.545455,-0.090114,0.250201,-1.087164,-0.16428,0.040339,0.24161,0.489621,-0.170679,0.086762,0.127898,0.156879,-0.090167,-0.420086,-0.372541,0.246916,0.03318,-0.101765,-0.071474,0.091065,0.447344,0.222918,-0.053832,0.373127],[0.29459,-0.452738,-0.633578,-0.361315,0.032905,0.519962,-0.372249,-0.359748,0.07424,-0.316428,0.564785,-0.939947,0.045408,0.264923,0.054147,0.430589,0.235577,-0.31392,-0.053279,-0.168415,0.564902,-0.402131,0.116464,-0.578709,-0.198141,-0.234593,0.443822,-0.008605,-0.553715,0.085717,0.053002,0.300217,-0.417554,0.326088,0.605082,-0.081575,-0.082884,-0.014904,0.185737,0.201629,-0.037351,-0.381756,0.472031,-0.570511,0.627503,-0.20529,0.508071,0.503935,0.241265,-0.026163,0.102124,0.150851,0.224853,0.576266,-0.263407,0.040819,-0.150566,0.523831,-0.666474,-0.260611,-0.356806,-0.512062,-0.110641,0.506999],[-0.148048,-0.227773,0.160304,-0.086485,0.177865,-0.059045,0.034515,0.329527,0.198079,0.478361,0.144591,0.010562,0.399394,-0.234439,0.073872,-0.863951,-0.298776,0.332563,-0.283672,-0.098282,0.174189,-0.056142,0.01779,0.270646,-0.110278,-0.209384,0.047866,0.185587,0.016906,0.23211,-0.317208,-0.075312,-1.177169,0.223099,-0.005827,-0.440265,-0.066429,-0.214299,0.054934,0.35406,-0.321147,0.574418,0.248793,-0.064429,0.631806,-0.216562,0.002214,-0.049146,-0.04844,-0.407411,0.086668,-0.034912,-0.582385,0.109194,0.541757,0.074586,-0.134054,-0.095988,-0.021315,-0.007478,0.074036,-0.058765,-0.112953,-0.007152],[0.345906,0.122214,-0.27742,0.252903,0.320674,-0.040154,0.162187,-0.159202,0.004704,-0.015666,0.325485,-0.169498,-0.690033,0.425918,-0.010067,0.118427,-0.325573,-0.096566,-0.611644,-0.042912,0.19773,-0.131072,-0.02322,0.214317,0.12318,0.17122,-0.191765,0.12929,-0.096027,-0.510941,-0.58688,0.074436,-0.560249,0.38916,0.392871,-0.364283,-0.122873,0.044627,-0.414593,-0.193538,-0.6824,-0.05204,-0.207166,0.090393,-0.166091,0.259229,-0.721256,0.232058,0.361734,-0.268799,0.049349,0.077873,0.083726,0.217299,-0.125211,-0.071853,-0.090874,-0.021761,-0.126767,0.105225,0.097148,-0.019977,0.138191,0.026642],[-0.022668,-0.118389,-0.360977,-1.058359,-0.167144,0.277598,-0.388647,-0.031609,-0.440897,0.277488,0.25745,-0.113264,-0.457066,0.101887,0.262342,-0.337141,-0.046874,-0.351617,0.238212,0.419371,0.106793,-1.101788,-0.020652,-0.30261,0.036675,0.218094,0.175867,0.093025,0.053618,0.440405,-0.364389,0.095731,-0.010076,-0.15524,0.763074,0.266738,-0.137812,-0.857387,-0.182713,-0.187023,-0.412687,0.157294,0.36752,0.187312,0.416315,-0.168847,0.17307,-0.321561,0.555497,0.01952,-0.079415,0.030917,0.177591,0.253247,0.203014,0.001653,0.194354,0.112242,-0.525957,0.175826,-0.10004,0.221226,0.873351,-0.381576],[-0.177685,0.520623,-0.182382,0.251495,-0.307274,0.481475,-0.321501,-0.055195,0.160068,-0.067927,-0.301163,-0.200629,0.335133,0.007991,0.104541,0.161581,-0.276021,0.333185,-0.207835,-0.584979,0.064517,0.341321,0.231685,0.372383,0.149108,-0.223753,-0.089935,0.089004,0.057241,-0.264067,0.073393,-0.049081,-0.631717,0.119969,0.135654,-0.197991,0.344743,0.210766,-0.403875,0.271284,-0.072412,-0.332098,0.278373,0.685213,0.314593,0.170968,0.119619,-0.151414,-0.235275,0.349796,-0.22472,-0.049424,-0.077922,0.196548,-0.336683,-0.16031,0.182115,0.49536,0.139988,-0.063105,-0.004884,-0.506686,0.422454,0.62865],[-0.38402,-0.187418,0.36931,0.211333,0.460967,-0.33797,0.392966,-0.112361,-0.043755,-0.08384,-0.108515,0.359151,-0.196876,-0.050548,0.05161,0.172907,0.192876,0.525487,0.315402,0.113302,-0.219461,-0.158623,-0.020707,0.401137,0.121793,-0.076455,-0.516056,-0.4007,0.09131,-0.152216,0.232409,-0.639896,-0.233717,0.379367,-0.752286,-0.361582,0.26863,0.445609,-0.432662,-0.145955,-0.055405,-0.26864,-0.259905,0.080192,-0.458687,-0.11705,-0.103383,0.245069,-0.452778,-0.261255,0.142841,0.122188,-0.026193,0.056546,-0.272878,-0.278079,-0.106382,0.121324,0.410761,-0.066272,0.027929,-0.47383,-1.207393,-0.010699],[-0.449144,-0.16806,0.557644,-0.253613,-0.171438,-0.19268,0.122736,-0.097738,0.218172,-0.018126,-0.093667,0.639808,-0.093475,0.132868,0.238458,-0.329992,-0.579503,0.093088,-0.06209,0.19411,-0.125664,-0.158656,-0.072029,-0.059651,0.073142,0.051629,-0.314384,0.583108,0.348658,0.202028,0.095893,-0.381136,-0.780177,0.403974,-1.006092,-0.119806,-0.088064,0.06439,0.036853,0.574617,-0.326156,-0.243611,0.003938,-0.182932,-0.23826,0.061743,-0.293767,0.338743,-0.324769,-0.20457,0.196085,-0.022556,-0.02676,-1.248997,0.118578,-0.23813,-0.422463,-0.155883,0.228565,-0.25164,0.146571,0.080099,-0.307213,-0.092306],[-0.124319,0.147829,0.126427,-0.35471,0.1527,-0.425584,0.329059,0.015173,0.277203,-0.070943,-0.150312,0.35484,0.16189,-0.515245,0.158772,-0.065033,-0.207325,0.365702,0.350703,0.197671,0.171596,0.194693,-0.411812,-0.70686,0.039643,0.394714,0.06332,0.072647,0.097478,-0.030523,0.473315,-0.069333,0.272209,-0.16839,-0.339347,-0.222401,0.49922,-0.272381,-0.489607,-0.46482,-0.158533,0.21491,-0.266105,-0.888402,-0.109691,0.176028,-0.530583,-0.057027,-0.113817,0.082307,0.215715,0.038347,-0.277347,-0.479683,-0.124732,0.093565,-0.021526,-0.396931,0.19528,0.354219,-0.342186,0.325476,-0.666313,-0.586585],[-0.352732,-0.037783,-0.213509,-0.186369,-0.480002,-1.296352,-0.210847,0.170757,-0.249726,0.254959,0.27807,-0.217366,-0.676126,-0.684585,-0.15447,-0.268355,-0.54521,-0.24674,-1.329602,-0.013051,-0.566462,0.27222,-0.099335,-0.513029,-0.355552,-0.432992,0.566182,0.60149,-0.008768,-2.893354,-0.929182,-0.273081,0.29855,-0.689804,0.238986,-0.602529,-0.64199,0.264934,0.22324,-0.509103,-0.895467,0.064079,-0.014987,0.634068,-0.249329,-0.785956,-0.376618,-0.047962,0.394514,-1.158493,0.031543,-1.471955,0.367391,0.330709,0.691734,-0.266371,-0.610245,-1.009377,0.416364,-0.247494,-0.263732,0.03796,-1.094455,0.268945],[-0.101875,-0.063947,-0.511198,-0.271153,-0.443422,-0.514026,0.005331,0.159902,-0.323394,0.535347,0.168248,-0.2957,-2.150842,-0.579727,-0.149916,0.121818,0.023455,-0.272126,-0.384077,0.052024,0.538049,0.315088,0.509469,-0.491056,0.012718,0.222442,0.071218,-0.096138,0.176098,0.493169,0.004864,0.350383,-0.098472,0.089804,0.423144,-0.267107,0.35026,-0.043957,-0.162653,0.346506,-0.695533,-0.01942,0.014253,-0.29247,0.154188,-0.269377,-0.366861,-0.149065,0.50828,-0.215487,0.03687,-0.842654,-1.047774,-0.230672,0.358483,-0.141199,-0.08469,-0.493589,0.00648,-0.138237,-0.043163,-0.41464,0.499618,-0.070106],[-0.281278,0.077585,0.430209,0.218267,0.385054,-0.988906,0.271522,-0.126971,0.309942,0.569333,0.282845,0.627091,-0.611853,-0.025899,-0.139638,0.343055,-0.012953,-0.270819,0.67208,-0.143933,0.058133,0.001125,0.571916,-0.203421,0.152912,0.101253,-0.522353,0.031516,0.555041,0.004699,-0.055143,-0.569636,0.322397,-0.044714,-1.144979,-0.112831,0.399246,0.381004,0.038543,-0.580794,0.401834,-0.133594,-0.675805,0.084957,0.138432,-0.22478,-0.343141,0.03295,-0.164203,0.013389,-0.215849,0.166856,-0.28723,0.082972,0.165115,-0.340674,0.195884,-0.828677,0.132082,-0.130835,0.254352,-0.256559,-0.018872,-0.206868],[-0.040286,0.518385,-0.087325,-0.049536,-0.439326,-0.534808,0.007853,-0.021986,0.089659,-0.290792,0.113075,0.482937,0.195816,0.418723,0.094706,0.336055,-0.096119,0.132232,0.047311,-0.008631,-0.030203,0.103054,0.284919,0.395766,0.057912,0.152192,0.054266,-0.380591,0.512023,-0.50572,0.136452,-0.09744,-0.512375,-0.045646,-0.659321,0.171926,-0.110207,0.244157,0.501672,0.013547,0.201743,-0.140111,-0.351447,-0.204943,0.035931,0.051978,0.249031,-0.271772,0.053712,-0.581163,0.243338,-0.280763,-0.256981,-0.239882,0.692608,0.123837,-0.089079,-0.585706,0.091203,0.004391,-3.090405,-0.275268,0.388565,0.066222],[0.042986,0.594391,0.508327,0.3155,-0.195743,-0.435104,0.262239,-0.421277,-0.013561,0.128542,0.028855,0.491319,-0.752689,0.125208,-0.693046,-0.338785,0.199482,0.037318,-0.230059,-0.204088,-0.152159,0.081353,0.047295,0.204931,0.184764,0.372216,-1.651556,-0.438641,0.260149,-0.298114,0.256333,-0.173801,-0.095423,-0.14043,-0.368008,0.20711,0.125983,0.443359,0.219624,-0.541577,0.064838,-0.395092,-0.30077,-0.060492,0.018088,-0.197473,-0.52692,-0.112681,-0.140524,-0.237576,-0.162862,0.370151,-0.151351,0.181161,-1.05858,-0.039143,0.001724,-0.564281,0.162416,0.407436,0.183275,-0.054862,0.022414,-0.618456],[-0.039693,0.055304,0.084584,0.35276,0.001752,0.13646,-0.069284,-0.518276,0.016972,-0.070385,-0.326481,0.17182,0.376729,0.166323,-0.069064,-0.092427,-0.187825,0.53612,0.152145,-0.129881,-0.157819,0.541119,-0.133985,-0.525956,-0.065337,0.168848,-0.133515,0.128529,0.026963,-0.050482,0.110507,-6.6e-05,-0.022573,0.399582,-0.182071,0.467262,0.234137,0.101426,0.495723,0.093687,0.282826,0.321206,-0.131653,0.293606,-0.163507,-0.205148,-0.570151,-0.455078,-0.150352,-0.786339,0.101589,0.0486,-0.282194,-0.353849,0.138255,-0.009722,-0.051677,0.121812,-0.01205,-0.049998,0.030082,-0.991657,0.741996,0.233215],[0.024028,-0.330438,-0.067179,-0.501905,-0.061042,-0.222767,0.052032,0.070624,0.210686,0.130058,-0.151488,0.113764,-0.707245,-0.140348,-0.156245,0.11365,0.560279,0.344558,0.370888,0.1687,-0.155011,0.095374,-0.120222,0.027832,0.209906,-0.132929,0.176536,-0.05247,0.116042,0.189246,0.210887,-0.020163,-0.492199,0.815016,0.088867,-0.001084,0.349006,0.151985,-0.148891,-0.220862,-0.12841,0.07917,0.070427,-0.044405,-0.024306,-0.412251,0.129458,0.116255,0.10815,0.097386,-0.056073,-0.049261,-0.231306,0.202193,-0.60019,-0.081243,0.378514,-0.236452,0.118339,0.100217,-0.209125,0.229762,0.279912,0.134013],[0.025327,-0.083389,0.082183,-1.228412,0.265844,-0.425779,0.112562,0.056585,0.10035,0.068702,0.203228,0.298854,0.479777,0.167795,0.080698,-0.194079,0.083446,0.114497,0.320341,-0.011956,0.119156,-0.037154,-0.183301,-0.450819,-0.046582,-0.176433,0.087184,0.031175,0.216489,-0.419994,0.398278,-0.035593,-0.745154,-0.194419,-0.342762,-0.409939,0.190478,-0.028173,-0.136506,-0.158531,-0.622909,-0.066901,-0.087901,0.082935,0.453987,0.250851,-0.698423,-0.120484,-0.108735,0.393951,0.024306,-0.066309,0.118254,-0.070006,-0.071138,0.103959,0.245645,-0.490418,-0.061127,0.120859,-0.437665,-0.072333,0.113049,-0.024449],[0.342308,-0.229076,-0.272008,0.090871,-0.68127,0.248849,-0.114815,0.361528,-0.25232,0.303641,0.221797,-0.074887,-0.198628,-0.17387,0.250296,-0.259619,0.458388,0.080724,-0.489877,0.091174,-0.140605,0.281363,-0.066299,0.114473,-0.215459,-0.186587,-0.250857,-0.449865,-0.025375,-0.158189,-0.478055,0.324367,0.044765,-1.023557,0.33322,-0.037812,-0.319619,-0.162863,-0.190609,-0.28057,0.257307,0.199914,0.21828,0.022217,-0.634731,-0.376071,0.235774,0.260754,-0.127113,-0.035236,0.244185,-0.760785,-0.148912,0.248207,-0.113926,0.357134,0.299116,-0.010539,-0.209193,-0.183808,-0.19557,-0.373249,-0.450845,0.417053],[-0.128304,-0.203069,0.162019,-0.03083,0.200751,-0.039848,-0.014111,0.45939,-1.176953,-0.236014,-0.296378,0.155496,0.190909,0.062806,0.098968,0.546264,-0.306761,0.054825,0.342432,0.093729,-0.007785,-0.203682,0.115539,-0.064902,0.038404,0.142835,0.37743,0.272212,0.04422,-0.201024,0.060432,-0.071151,0.866999,0.40894,-0.206513,-0.449958,0.069636,-0.279172,-0.425206,-0.135245,0.446712,-0.035823,0.117549,-0.033338,0.298398,-0.111406,-0.301466,-0.030254,-0.125226,-0.127961,-0.081235,0.178404,-0.762807,-0.629084,-0.116709,0.07309,0.366739,-0.049799,-0.014759,0.111783,-0.048018,-0.198862,0.15638,-0.091703],[-0.637342,0.197289,0.432851,0.192986,-0.021782,-1.17827,0.37512,0.114836,0.23695,0.31508,-0.161424,0.506213,0.151004,-0.264885,0.051111,-0.293357,0.726975,0.14416,-0.539039,0.338129,0.170567,-0.289404,-1.058177,-0.791576,0.011557,0.537918,-0.267875,0.066388,0.217824,0.142368,0.289442,-0.110241,-0.163586,-0.069944,-0.897809,0.242661,0.398935,-0.471086,0.021565,-0.539527,0.332593,-0.123926,-1.08494,0.200194,-0.323647,-0.210758,0.343485,0.50467,-0.436229,-0.002188,-0.118426,0.496309,-0.227824,-0.165999,0.515078,0.124176,0.17379,-1.081738,0.531562,0.106798,0.342206,0.54586,0.287287,-0.520275],[-0.217207,0.129522,0.317107,0.19617,0.624061,-0.601013,-0.120305,0.061942,-0.606605,-0.263174,-0.066888,0.467713,0.538507,-0.669394,0.197385,-0.006973,0.101707,0.048563,0.295491,0.179659,0.058526,0.180455,0.505416,-0.801219,0.069326,-0.571087,0.28855,0.213865,0.249093,-0.375546,-0.175696,-0.22371,-0.506125,0.12378,-0.854248,-0.335429,0.293003,0.051272,-0.284695,0.915708,0.192718,0.099432,-0.438629,0.136608,0.061712,0.02511,-0.829365,0.03202,-0.114979,0.07347,0.210921,0.143006,-0.335267,0.072891,0.072695,0.004202,0.352506,-0.576724,0.01945,-0.108125,0.670499,0.442861,-2.073131,0.422607],[0.312891,0.480296,-0.653326,-0.075247,0.196382,0.088347,-0.154259,-0.223339,0.260846,0.254978,-0.274497,-0.088803,-0.564278,0.044622,0.095113,-0.67477,0.312097,0.013576,0.506402,0.064152,-0.044915,-0.01488,0.063756,-0.161699,-0.217296,-0.269007,-0.351328,-0.497469,0.191764,0.016068,-0.166535,0.163675,-0.285431,0.206445,0.215727,0.421414,0.214456,0.002402,-0.454509,-0.1746,-0.365591,-0.044575,-0.049887,-0.109268,0.123932,-0.360006,-0.085317,-0.174536,0.468495,-0.807902,0.012612,-0.113175,-0.490084,0.086468,-0.315098,-0.05725,0.086684,0.005946,-0.05399,0.444381,0.1358,0.152198,-0.010227,-0.444683],[0.051576,-0.310899,0.068232,-0.384525,0.131617,0.276083,-0.041565,0.187128,-0.01315,-0.429562,0.284669,-0.442067,-0.471406,0.261973,-0.111277,0.040035,-0.60523,-0.587872,0.207506,0.101274,0.086646,0.35237,0.035458,-0.456811,-0.183333,-0.066103,0.080731,-0.534799,-0.31769,-0.130723,-0.065056,0.157979,0.281544,-1.080998,0.423011,-0.128254,-0.272561,0.03591,-0.092288,0.126635,-0.094713,-0.27464,0.061754,-0.278863,-0.051043,0.055703,0.566361,0.059273,0.156521,-0.013473,-0.122694,0.295109,0.058663,0.232433,-0.078211,0.128885,0.17743,0.321609,-0.128624,-0.31722,0.496141,0.25076,0.040483,-0.020143],[-0.046133,0.187779,-0.034719,-1.02543,0.368991,0.195709,0.036978,0.175192,-0.060509,-0.334872,0.179849,0.047189,-0.73727,0.292107,0.113107,0.317926,0.035382,0.139452,-0.078816,0.022671,-0.305833,-0.065751,-0.050951,-0.267403,0.157974,0.0563,-0.251834,-0.770597,-0.093058,0.283809,0.069523,0.121416,-0.142904,0.129518,-0.011583,0.310158,-0.391445,-0.197358,-0.108966,0.072561,-0.465378,-0.003294,-0.008849,0.384298,-0.742205,0.071101,0.413383,-0.381763,-0.067089,0.336534,-0.245154,0.001314,0.23954,-0.164274,-0.04845,0.12,0.103275,0.210344,-0.043747,0.011615,-0.161301,0.240904,0.203743,0.437059],[0.067726,-0.090479,-0.056333,-0.568444,0.291367,0.265249,-0.034561,-0.220096,0.053528,0.248028,-0.714303,-0.069919,0.684841,-0.156674,-0.13525,-0.183408,-0.586112,-0.112328,-0.719338,0.307153,0.084917,-0.006203,-0.00771,-0.372809,0.002091,-0.295797,-0.519729,-0.658696,-0.075883,0.000551,-0.468017,0.173715,0.876901,0.261972,0.08965,0.081939,0.116626,-0.017078,0.392909,0.049324,0.410098,-0.542749,-0.008181,0.025429,0.040572,-2.038346,0.332529,0.167307,-0.054261,-0.103073,-0.018815,-0.149734,-0.067909,-0.590353,0.027832,-0.026117,-0.137645,0.241865,-0.014303,-0.187262,0.251895,0.344225,0.364209,0.174934],[-0.225934,0.480065,0.271369,-0.152463,-0.92964,-0.924011,-0.274525,-0.183186,0.405063,-0.119136,0.354398,0.519444,0.353883,-0.352207,0.112375,-0.04541,0.301704,0.808178,-0.79767,-0.018409,-0.031914,0.005984,-0.008728,0.214694,0.110812,-0.22863,-1.126412,-0.224321,0.504568,-0.082165,0.535065,-0.130755,-1.512455,0.17525,-0.382016,0.193122,0.607308,0.109993,-0.029989,0.449696,-0.006356,-0.584904,-0.76441,0.254353,0.356818,-0.265135,-0.14582,0.388015,-0.000691,0.004556,-0.142022,0.300251,0.96288,0.725106,0.407971,0.034108,-0.063138,-0.754651,0.354903,-0.029165,0.457388,0.366833,-0.786024,0.416053],[0.065502,-0.147917,-0.093093,-0.320156,-0.207768,-0.111847,-0.105148,-0.0472,-0.392264,-0.213279,-0.102534,-0.25202,-0.021177,0.208993,-0.087929,-0.21825,0.345403,-0.25681,-0.360455,-0.041892,0.19984,0.260443,0.277912,-0.044884,0.077502,0.759149,-0.649472,0.191101,-0.055168,0.122353,0.178328,0.362306,-0.675559,-0.055503,0.283531,0.040343,-0.368149,0.109209,-1.227873,0.066624,-0.362618,0.19628,0.03384,0.5215,-0.311347,-0.145672,-0.045917,0.133789,-0.012197,0.059589,0.112309,-0.26995,-0.57477,0.094841,-0.405903,-0.166994,-0.237973,-0.072553,0.122167,0.197617,-0.035704,-0.032354,0.016347,0.277666],[0.049507,-0.035596,0.171149,0.404507,0.597598,0.215008,-0.088221,-0.230076,0.191687,0.66463,0.16175,-0.098253,0.231283,0.266264,0.077929,-2.275735,0.40956,-0.182267,-0.09908,0.229336,-0.061792,0.184241,0.168379,-0.142122,-0.108143,-0.063763,-0.080395,0.433147,-0.034231,0.054438,-0.461924,-0.060427,0.08785,-0.032296,0.027653,0.092895,-0.390224,0.098425,-0.331468,0.055145,0.259546,0.011896,0.080729,-0.157182,0.095656,-0.052861,0.320278,-0.112984,-0.002024,-0.128971,-0.278046,0.093114,0.200457,-0.025853,-0.091958,0.034895,0.024928,0.192151,0.140875,-0.211151,0.519593,-0.162347,0.267008,-0.250652],[0.079863,-0.078438,-0.086167,-0.522437,0.145121,0.186476,-0.120026,-0.135976,0.172721,0.097699,-0.001235,-0.269763,-0.392423,-0.028303,-0.066461,-0.701835,0.202367,-0.437514,0.318424,-0.221304,-0.039867,-0.592688,0.255531,0.240175,0.104723,-0.009963,0.448124,0.131704,-0.010178,0.356542,-0.138961,0.099606,-0.129577,0.101803,0.250647,0.200092,-0.339183,0.020053,-0.124263,0.077296,0.154143,0.050886,0.285727,-0.792668,0.104884,-0.040303,-0.144339,-0.586055,-0.001336,0.076521,-0.009023,0.001512,-0.091959,0.353012,0.248132,-0.008433,-0.067416,0.131792,-0.12317,-0.105405,0.140229,0.047307,-0.558354,-0.301758],[-0.056054,-0.172634,0.16044,0.325889,0.48195,-0.357971,0.006599,-0.158825,0.020733,-0.614928,0.239903,0.203645,0.019955,0.271403,0.035126,0.788715,0.213809,0.247875,0.109593,-0.165374,-0.261982,0.172223,0.151732,0.013698,-0.090244,0.196307,-0.398264,-0.545031,0.174585,-0.464164,-0.228868,-0.17152,0.30867,0.490347,-0.2087,0.064826,0.134172,0.026774,-0.003021,-0.444175,0.346079,0.414721,-0.060189,0.659449,0.266267,0.064647,0.083551,-0.051929,-0.038592,-0.090078,-0.108903,0.105173,-0.437686,-0.018535,-0.053336,0.022195,0.296088,-0.330891,-0.073241,-0.425482,0.438166,-0.159777,0.047179,0.339401],[-0.08308,0.104325,-0.272455,-1.950601,-0.43001,-0.188415,-0.09063,0.177599,-0.176132,-0.15734,0.066596,-0.389944,0.697365,-0.050253,-0.223357,-0.003536,0.124673,-0.277968,-0.295101,0.022098,-0.277148,0.26392,-0.124924,0.327722,-0.323407,-0.002842,-0.62196,-0.020589,-0.038722,-0.4828,-0.165337,0.22124,0.341682,0.125583,0.507196,-0.190927,-0.777965,-0.110888,0.473139,-0.367718,-0.172044,0.1087,-0.141853,0.21181,0.43694,-0.296301,0.098041,0.073409,0.367868,-0.50243,-0.123602,-0.185956,-1.054042,0.128378,-0.06796,-0.174871,-0.063109,-0.039576,-0.120784,-0.380101,-0.77441,-0.028771,-0.01856,-0.014363],[-0.081826,-0.049594,0.124348,1.607159,0.028323,-0.202185,0.042957,-0.214299,0.495464,0.341276,-0.213371,0.089949,0.316281,-0.088296,0.073064,0.038734,-0.251094,0.02668,-0.208341,-0.15809,0.07957,0.513288,0.123022,-0.425835,0.014772,0.012218,-0.45938,-0.121346,0.066371,-0.676335,-0.02207,-0.111695,-2.405734,-0.123378,-0.144079,-0.069247,0.149993,0.149934,0.636068,0.085679,0.299263,0.305669,-0.131503,-0.833776,0.394583,-0.329282,-0.097869,-0.03767,-0.037371,0.258422,0.20447,-0.062211,-0.196524,-0.178388,-0.259035,0.111121,0.054142,-0.152298,0.137661,-0.042824,-0.301201,-0.100339,0.043152,0.101244],[-0.044945,0.00373,0.131259,-0.711189,-0.473613,0.138416,0.099735,0.062158,-0.096564,0.123994,-0.326881,0.171472,0.052431,-0.45,-0.04472,0.187359,-0.069812,0.357239,-0.430066,-0.069283,0.083043,0.344025,0.062589,0.148852,0.05796,-0.082159,-0.358984,0.221373,-0.081069,0.049958,0.428176,-0.044322,-0.431315,-0.032004,-0.141941,-0.283358,0.334113,0.002103,-0.634613,0.247932,-0.316142,-0.297272,0.148612,-0.751257,-0.943025,0.656008,-0.956349,-0.212125,-0.189016,0.874735,0.110538,0.303641,0.409858,0.251344,-0.621354,-0.281389,-0.016837,0.159476,-0.057763,0.073004,-0.531699,-0.574082,0.43043,0.077627],[-0.149503,-0.372508,-0.063114,0.236558,-1.46221,0.068361,-0.037396,-0.333256,-1.149491,-0.017478,0.199415,-0.053849,0.336172,0.396101,0.256118,0.449319,0.275928,-0.000844,-0.053383,0.04072,0.100263,0.302067,-0.013542,0.32836,0.263377,0.290255,0.061469,0.067937,0.13073,0.172753,0.34261,-0.033362,0.065286,0.085094,0.091738,0.643741,-0.293576,0.571026,-0.312649,0.171441,0.380726,0.469798,-0.103617,-0.176674,0.181316,-0.128165,0.351235,0.23524,0.129608,0.160986,0.040292,-0.154381,0.149805,0.389487,0.145901,0.31326,0.055573,0.159004,0.333067,-0.053643,-0.500828,0.327099,0.048048,-0.166243],[-0.060467,-0.071475,0.07853,-0.430473,0.332411,0.077677,0.27992,0.052277,0.397786,0.053181,0.060374,0.083757,-0.124422,0.241789,-0.014673,-0.062828,0.333489,0.23778,0.126841,-0.069758,-0.004876,0.306592,-0.262893,0.444581,-0.085623,0.335943,0.191875,0.170501,0.016594,0.428607,0.476526,-0.00943,0.324302,-0.315979,-0.022527,-0.42576,-0.016175,0.021912,-0.284552,0.20205,-0.102151,0.418638,-0.064469,-0.068392,0.482243,-0.489295,0.258663,-0.605718,0.027386,0.594666,-0.04225,0.136039,0.358455,-0.498938,-0.47514,-0.078988,-0.143211,0.086482,-0.019177,-0.159544,-0.493009,-0.156591,-0.016759,-0.216746],[-0.028595,-0.109655,-0.462829,0.054335,0.219851,-0.713087,-0.106764,0.072157,-0.271511,0.034102,0.039057,0.011868,-0.871211,0.648706,-0.22427,0.262199,0.066391,-0.164574,0.045211,0.327906,-0.062316,-0.615161,0.175592,-0.648972,-0.048148,0.459955,-0.309107,0.217861,0.440408,0.746276,0.21343,0.289324,-1.892764,0.063512,0.101506,0.152562,0.0138,-0.139035,-0.113044,0.172411,-0.532838,0.190513,-0.169064,0.030069,0.056783,0.421977,-0.290121,0.169054,0.136552,-0.172637,-0.175006,-0.639398,0.62338,0.032942,0.208378,0.228248,-0.078951,-1.015394,-0.047328,-0.101733,0.173009,-0.297277,0.149689,-0.295064],[-0.079508,-0.010142,-0.372775,-0.12937,0.026725,0.054049,0.035229,0.046825,-0.1093,0.362685,-0.265298,0.133396,-0.378771,-0.315664,0.004693,-0.121465,-0.056879,-0.092039,-0.224194,-0.087217,0.169995,-0.48698,-0.239866,-0.035391,0.048056,0.100933,0.024681,-0.68258,0.266474,-0.547803,0.177224,0.161073,-0.593571,-0.101681,-0.088704,0.46724,-0.019747,0.007349,-0.390767,-0.337924,-0.331485,-0.240603,0.122323,-0.085859,0.663566,-0.758733,-0.003944,-0.663169,0.184736,-0.110846,0.050378,-0.248616,-0.951869,0.445064,-0.250205,-0.023891,0.15217,-0.198015,-0.314369,0.434477,0.016286,0.198496,-0.208203,-0.163613],[0.054068,0.07852,-0.16024,-1.048355,-0.126537,-0.042164,-0.453593,0.021329,-0.099228,0.589729,0.493686,-0.300252,-0.912612,-0.357791,-0.066686,-0.175488,-0.142982,-0.031704,0.279332,0.053075,-0.079062,0.413038,0.229775,-0.898597,0.127375,-0.342434,-0.162406,-0.479506,-0.065692,-1.625368,-0.416366,-0.005918,1.035561,0.086579,0.310582,-0.297586,-0.592195,-0.011355,-0.506754,-1.039782,0.038138,0.258785,0.068634,-0.081398,0.176861,-0.73339,0.477185,0.364219,0.192741,-0.023676,-0.136185,-0.069781,-0.175429,-0.073854,0.236983,0.052184,0.077149,-0.011087,-0.070323,0.047855,-0.125429,0.087979,0.080641,-0.21353],[0.00836,0.187802,-0.14549,-0.401641,-0.190263,-0.183515,-0.395945,0.272054,0.228218,-0.126998,0.185842,-0.202043,-1.550215,0.179764,0.094929,-0.303066,-0.036753,0.01837,0.495081,0.037448,-0.117054,-0.2115,0.050378,-0.455257,-0.024294,-0.416109,0.161352,-0.265283,-0.123689,0.015836,-0.04643,-0.035276,0.104003,0.388774,0.110942,-0.168774,-0.139482,-0.021918,-0.23007,-1.238711,0.172963,0.285888,-0.006397,0.584968,0.116701,-0.424738,0.096321,0.375348,0.004418,0.628262,0.041368,-0.240071,0.044082,-0.227168,0.05646,0.105795,-0.35289,-0.05111,-0.001857,-0.223488,-0.025539,0.034422,0.244114,-0.646921],[0.245474,0.204804,0.04794,0.041749,0.301986,-0.527084,0.014978,-0.164848,0.240514,0.062643,-0.081427,0.240494,-0.542075,-0.018704,-0.052123,0.09961,0.019836,0.033478,-0.221718,-0.066311,0.179606,0.016873,-0.26748,0.082681,-0.021603,0.794529,-0.320215,-0.362007,0.250288,0.531066,0.321178,-0.042392,0.158857,0.000996,-0.233494,0.091954,-0.060414,-0.029827,-0.646024,-0.789744,0.021649,0.176418,-0.272988,0.356569,0.058913,0.121427,-0.662377,0.016529,0.141586,-0.101282,0.14641,0.2618,-1.135025,0.149363,-0.289481,-0.20398,0.164239,-0.655297,0.074498,0.278482,-0.965322,-0.236675,0.566845,0.076556],[-0.102268,0.0965,0.186029,-1.042161,0.190528,-0.190574,0.316688,0.195276,-0.353061,-0.708163,0.596409,0.230322,0.173598,0.079134,-0.029686,-0.11957,-0.106064,0.219699,-0.15141,-0.224901,0.028257,0.247218,0.425271,-0.657072,-0.009791,0.134548,-0.33939,0.369139,0.071622,0.029193,-0.388639,-0.106327,0.25193,0.301912,-0.181821,-0.460715,-0.11958,0.228224,0.000648,0.128622,-0.443208,0.116385,-0.003383,0.316212,-0.05878,0.312053,-0.553365,0.300881,-0.098065,-0.082305,0.032005,-0.112203,0.55186,0.080579,-0.077381,-0.104878,0.14705,-0.154983,0.064748,-0.022617,0.58069,-0.368655,0.259884,-0.079174],[0.425815,-0.144862,-0.481813,-0.025387,-0.322426,0.089548,-0.070865,-0.036592,-0.099883,-0.696827,0.349393,-0.167717,0.522937,0.155767,0.116927,-0.216967,-1.17969,-0.019305,0.178533,0.114752,-0.01785,0.324321,0.384597,-0.181889,0.256314,-0.239172,0.792479,0.594227,0.221173,-0.427458,0.00832,0.143211,0.25914,-0.094155,0.32991,-0.997796,-0.652465,0.044827,-0.423113,-0.500217,0.073105,0.472689,0.113487,-0.199407,0.004823,0.513259,-0.253988,0.265402,0.477183,-1.298125,0.062593,-0.057053,-0.378155,0.332643,0.540947,-0.157258,0.106877,-0.088705,-0.22965,-0.311881,0.357617,0.113904,-0.063307,0.202495],[-0.066032,-0.244301,0.073728,0.075092,0.042062,-0.080783,0.055887,-0.106294,0.162261,0.252558,-0.39259,-0.232952,-0.45506,-0.092084,-0.279537,-0.09902,0.005907,-0.062108,-0.18309,-0.026311,0.042727,-0.154152,-0.237331,-1.058294,-0.208972,-0.181167,0.074768,-0.008789,-0.207808,0.314622,-0.336468,-0.022127,-0.125667,0.04408,0.291943,-0.007753,-0.057799,-0.135885,0.645552,-0.192578,0.428368,0.06208,0.366429,0.141216,0.247882,0.138373,-0.447782,-0.856605,0.03564,-0.013904,0.080884,-0.121684,0.308208,-0.032192,0.05011,0.000659,-0.143164,-0.058135,0.00306,-0.007599,-0.475382,-0.685373,-0.057854,-0.234392],[0.042276,0.106569,-0.03116,0.437388,-0.563973,0.145607,-0.02213,-0.655439,-0.194714,0.284103,-0.837979,-0.132439,0.065389,0.109162,-0.178982,0.581476,-0.303823,-0.218014,0.649637,-0.141515,0.004297,0.059201,0.231586,-0.109625,0.082695,0.17539,0.59981,0.151567,-0.033978,0.071879,0.549991,0.034525,-0.002813,0.050041,0.261661,0.046718,0.285091,-0.302269,0.455269,-0.000478,-0.147472,-0.151965,0.164917,-0.253322,-0.463413,0.858637,-0.330053,-0.320782,-0.034205,0.522917,-0.227767,0.068903,0.032092,0.177214,0.135087,0.051765,0.069744,0.142061,0.057623,0.14359,0.178339,0.061627,-0.518139,-0.083215],[0.04406,-0.234513,-0.049586,0.607057,-0.115986,0.386773,0.076927,-0.121042,0.316516,0.180954,-0.285538,-0.202318,0.638428,0.108755,0.031521,0.345946,0.141845,0.10835,-0.073709,-0.057272,-0.054698,0.022742,-0.231341,-0.243428,-0.042445,-0.275048,-0.507825,0.098971,-0.168731,0.10977,0.438161,0.071039,-0.451539,0.448525,0.31634,0.683487,-0.089707,0.023403,0.341221,-0.50754,0.370024,0.386644,0.172136,0.08604,0.090801,-0.197595,-0.401656,0.110887,-0.001228,-0.215257,0.020813,0.007978,0.083789,0.006106,0.353021,0.012853,-0.125332,0.43003,0.047822,-0.119133,0.305896,0.673123,-0.052252,0.456253],[0.090055,-0.650785,-0.272004,0.206307,-0.258993,0.409855,-0.186075,-0.03352,0.299845,-0.229455,0.076653,-0.655064,-0.219709,0.125179,-0.175354,0.226727,-0.058471,-0.647819,-0.23445,0.041843,0.301482,-0.175337,0.246015,0.420336,-0.242234,-0.337978,0.001858,0.290233,-0.283206,0.202349,0.057025,0.249733,0.541522,-0.192836,0.715241,-0.201626,0.085243,-0.505648,0.298796,-0.678918,-0.217748,0.146232,0.318787,-0.40729,0.18619,0.525355,0.346886,0.04915,0.178446,0.58422,-0.05689,-0.016912,0.434348,-0.095091,-0.262904,-0.023207,-0.109733,0.432088,-0.253793,0.117294,0.25111,-0.135007,-0.99327,-0.068346],[-0.115034,-0.068434,-0.212799,-0.688958,-0.162075,-0.143902,0.106298,-0.098283,-0.172857,-0.410376,0.108656,0.085277,-0.066542,-0.053476,-0.088574,-0.576699,0.046836,-0.318376,-0.013082,0.324523,-0.352784,0.166698,0.133537,-0.006696,-0.080262,-0.555841,0.450213,0.101969,0.142542,-1.130505,-0.23327,0.0167,-0.349175,-1.069838,-0.023067,0.200161,-0.262643,-0.115402,0.052057,0.207874,0.157718,0.18885,0.014629,0.28974,0.469318,-0.986624,0.090589,-0.083919,0.392515,0.384258,0.305898,0.150923,-1.295357,0.241655,-0.200231,-0.028643,-0.11293,-0.135367,-0.127832,0.049514,0.320529,-0.123253,-0.55761,0.169866],[0.050351,-0.171905,0.014836,-0.067188,-0.497553,-0.294028,-0.057982,0.642558,-0.342497,0.089536,-0.069736,0.019542,0.264454,-0.513569,0.453037,-0.253212,0.412897,-0.567037,0.145798,-0.123057,-0.103418,0.172805,-0.052694,0.752884,0.074497,-0.800689,-0.325394,0.099824,0.212612,-1.411493,0.231393,-0.061581,-0.087212,-0.13904,-0.169327,0.442484,-0.446601,-0.362594,-0.707842,0.377408,0.089979,0.337212,0.001724,-0.125535,-0.078074,-0.439251,0.241931,0.356044,-0.247991,0.344002,-0.109052,0.126362,-0.433613,0.067892,0.42757,0.050522,-0.277886,-0.328591,0.197355,0.221636,0.039604,0.4168,-0.085784,-0.460534],[-0.019021,0.250258,-0.008156,-0.41599,-0.845234,-0.169005,-0.062589,-0.45857,0.18983,-0.640179,0.242423,0.021753,-2.609392,-0.056183,0.153574,0.282634,0.276675,-0.021296,-1.118435,0.098867,-0.192484,-0.090317,0.01128,-1.263277,-0.068275,0.216912,-0.76066,0.257517,-0.0362,-0.546084,0.343634,-0.037698,0.217136,-0.384435,-0.070572,-0.140189,-0.100849,-0.213338,0.190448,0.056099,0.421723,0.381171,-0.30761,0.089977,-0.198716,-0.113742,0.465542,0.177857,0.119702,-0.221117,-0.165497,0.207948,-0.126472,0.157273,-0.135003,0.035708,-0.175326,-0.085308,-0.084624,0.010891,-0.156067,-0.234637,-0.011727,0.29116]],"b":[0.327075,0.457383,0.245435,-0.303584,-0.022966,-1.109971,0.246723,-0.170989,-0.022521,0.387376,-0.221021,-0.215731,-0.962384,-0.208517,-0.224683,-0.004932,0.338219,0.476569,0.427565,0.165218,-0.157219,-0.485686,-0.41943,-0.346721,0.375403,0.314129,0.220602,-0.502449,0.558074,0.13014,0.235698,0.482475,-0.593164,-0.14068,-0.321867,0.465982,0.331152,0.151147,-0.289927,-0.320883,-0.072997,-0.166362,0.346098,-0.022432,0.193415,-0.242821,-0.335,-0.340366,0.625315,-0.098412,0.070253,-0.173367,-0.381726,-0.140843,-0.372241,0.453854,0.26834,-0.107601,-0.51364,0.459248,-0.312695,-0.189409,0.078519,-0.124642]},{"W":[[0.026856,-0.206849],[0.10175,0.092945],[0.117827,0.305043],[2.079666,0.002482],[0.486266,0.001221],[-0.058243,0.573949],[0.065565,0.119033],[-0.156114,-0.009937],[0.203373,0.001482],[0.149979,-0.002371],[-0.160196,0.002364],[-0.008361,-0.438852],[1.005116,0.0187],[-0.200226,0.031846],[0.039715,0.059971],[0.363914,0.000908],[-0.273071,-0.011086],[0.220471,0.068769],[-0.509749,-0.002048],[-0.001528,0.128285],[0.049491,-0.09043],[-0.172087,-0.000992],[-0.242963,0.037834],[1.292006,0.006902],[0.012896,0.080211],[0.282794,0.0103],[0.504597,-0.020324],[-0.380203,-0.00876],[0.021535,0.262302],[-0.593897,-0.005848],[0.315941,0.008696],[0.059547,-0.260668],[1.32902,0.011468],[0.359169,-0.000289],[-0.307809,0.35045],[-0.29198,-0.006534],[0.328978,0.0538],[-0.095748,0.057795],[0.845961,0.00254],[0.928735,0.002326],[-0.335087,-0.006391],[-0.201925,-0.007935],[0.122037,-0.189536],[0.366167,-0.005667],[-0.234465,-0.004596],[-0.420097,-0.010044],[-0.801204,0.006149],[-0.398515,0.001057],[-0.001871,-0.247716],[-0.331507,-0.000938],[0.009851,0.086657],[-0.114389,-0.070918],[2.221841,0.005257],[0.336398,0.00102],[-0.409203,-0.005807],[0.001727,-0.125692],[0.047012,0.036797],[-0.064659,-0.612001],[-0.033278,-0.198869],[0.005066,0.075613],[0.354274,-0.004614],[-0.239076,-0.002696],[-0.756745,-0.001509],[-0.281967,0.001111]],"b":[0.145467,0.202408]}]}'

In [ ]:
# VYHODNOTENIE A MODELY-PORADCOVIA (src/walkforward.py)
"""Vyhodnotenie poradcu na histórii: pre každý počiatočný rok XY model odpovie len z dát známych do XY.

Pre každý XY a horizont H:
  1. skutočný víťaz  = ktorý scenár mal po H rokoch väčší majetok (model.tier2_grid, skutočné dáta XY+1..XY+H),
  2. odpoveď modelu  = „kúpa“ / „prenájom“ vypočítaná z dát <= XY,
  3. porovnanie      = podiel rokov, v ktorých mal model pravdu (spolu so základnými stratégiami
                        „vždy kúpa“, „vždy prenájom“).

Modely:
  odhad2 – ten istý, ale polovica odhadu sa vracia k dlhodobému priemeru (MEAN_REVERSION).
  odhad  – „sústava malých modelov“ (poznámka 1B): z posledných L rokov histórie odhadne rast cien,
           rast nájmov a výnos akcií, nasimuluje oba scenáre s týmito konštantnými hodnotami a porovná.
  sieť   – neurónová sieť z webu, dostane tie isté odhadnuté vstupy (orezané na rozsah, v ktorom sa učila).
Nič z budúcnosti XY nevstupuje do odpovede (okrem parametrov siete – viď správa, kapitola o obmedzeniach).
"""
import json
import numpy as np
import pandas as pd


LOOKBACK = 20                         # dĺžka okna histórie pre odhad (určená vopred, nelaďovaná na výsledkoch)
HORIZONS = (10, 20, 30)
MEAN_REVERSION = 0.5                  # druhý model: polovica odhadu sa vracia k dlhodobému priemeru (zvolené až po prvom behu, viď správa)
RATE_SPREAD = 0.015                   # marža hypotéky nad dlhodobou sadzbou (rovnako ako v tier2_grid)


def known_inputs(d: pd.DataFrame, year: int, lookback: int = LOOKBACK, shrink: float = 0.0) -> dict:
    """Odhady z dát <= year. d: ročná tabuľka s indexom year (stĺpce housing_capgain, housing_rent_yd, eq_tr, ltrate).

    shrink = 0: čistý priemer z posledných `lookback` rokov. shrink = w > 0: podiel w sa „vráti k dlhodobému
    priemeru“ (všetky roky v dátach až do `year`, od roku 1901) – ochrana proti extrapolácii boomu."""
    full = d.loc[:year]
    h = full.tail(lookback + 1)
    geo = lambda x: float(np.exp(np.log1p(x).mean()) - 1)

    def est(df):
        g = df["housing_capgain"].iloc[1:]
        r = (df["housing_rent_yd"] / df["housing_rent_yd"].shift(1) * (1 + df["housing_capgain"]) - 1).iloc[1:]
        return geo(g), max(geo(r), 0.0), geo(df["eq_tr"].iloc[1:])
    g, rent_g, eq = est(h)
    if shrink:
        g2, r2, e2 = est(full.dropna(subset=["housing_capgain", "housing_rent_yd", "eq_tr"]))
        g, rent_g, eq = [(1 - shrink) * a + shrink * b for a, b in ((g, g2), (rent_g, r2), (eq, e2))]
    return {
        "house_growth": g, "rent_growth": rent_g, "equity_return": eq,
        "gross_yield": float(h["housing_rent_yd"].iloc[-1]),
        "mortgage_rate": max(float(h["ltrate"].iloc[-1]) / 100 + RATE_SPREAD, 0.01),
    }


def advise_estimate(inp: dict, years: int, p=None) -> float:
    """Model „odhad“: výhoda kupujúceho po `years` rokoch pri konštantných odhadnutých hodnotách (> 0 = kúpa)."""
    from dataclasses import replace
    p = replace(p or SPAIN, gross_yield=inp["gross_yield"])
    n = years
    adv = simulate(np.full(n, inp["house_growth"]), np.full(n, inp["rent_growth"]),
                   np.full(n, inp["equity_return"]), np.full(n, inp["mortgage_rate"]), p, 1, n)
    return float(adv[-1])


# ------------------------------------------------------------------ neurónová sieť (numpy)
RANGES = {"gross_yield": (0.01, 0.07), "house_growth": (-0.05, 0.20), "rent_growth": (0.0, 0.20),
          "equity_return": (0.0, 0.20), "mortgage_rate": (0.01, 0.20)}   # záloha; skutočné rozsahy sú v súbore s váhami


def load_net(path_or_dict):
    w = path_or_dict if isinstance(path_or_dict, dict) else json.load(open(path_or_dict))
    return {k: (np.array(v) if k in ("mu", "sd") else v) for k, v in w.items()}


def advise_net(net: dict, inp: dict, years: int, p=None):
    """Vráti (P(kúpa vyhráva), počet vstupov, ktoré bolo treba orezať na tréningový rozsah)."""
    p = p or SPAIN
    clipped = 0
    v = {}
    for k, (lo, hi) in net.get("ranges", RANGES).items():
        v[k] = min(max(inp[k], lo), hi)
        clipped += v[k] != inp[k]
    x = np.array([0, years, p.ltv, v["gross_yield"], p.buy_cost, v["house_growth"], v["rent_growth"],
                  v["equity_return"], v["mortgage_rate"]], float)
    a = (x - net["mu"]) / net["sd"]
    for i, L in enumerate(net["layers"]):
        a = a @ np.array(L["W"]) + np.array(L["b"])
        if i < len(net["layers"]) - 1:
            a = np.maximum(a, 0)
    return float(1 / (1 + np.exp(-a[0]))), int(clipped)


# ------------------------------------------------------------------ vyhodnotenie
def evaluate(annual: pd.DataFrame, net: dict | None = None, horizons=HORIZONS, first=1950, lookback=LOOKBACK):
    """annual: tabuľka ako spain_annual_1870_2025.csv. Vracia DataFrame: jeden riadok na (rok, horizont)."""
    actual = tier2_grid(annual, start_from=first, max_years=max(horizons))
    d = annual.set_index("year").sort_index()
    rows = []
    for H in horizons:
        for y in actual.index:
            if H not in actual.columns or pd.isna(actual.loc[y, H]):
                continue
            inp = known_inputs(d, int(y), lookback)
            adv_est = advise_estimate(inp, H)
            adv_mr = advise_estimate(known_inputs(d, int(y), lookback, MEAN_REVERSION), H)
            row = {"start": int(y), "H": H, "actual_adv": float(actual.loc[y, H]), "buy_won": bool(actual.loc[y, H] > 0),
                   "est_adv": adv_est, "est_buy": adv_est > 0, "mr_adv": adv_mr, "mr_buy": adv_mr > 0, **{"in_" + k: v for k, v in inp.items()}}
            if net is not None:
                pb, clipped = advise_net(net, inp, H)
                row.update(net_p=pb, net_buy=pb > 0.5, net_clipped=clipped)
            rows.append(row)
    return pd.DataFrame(rows)


def summarize(ev: pd.DataFrame) -> dict:
    out = {}
    for H, g in ev.groupby("H"):
        s = {"n_years": int(len(g)), "first": int(g.start.min()), "last": int(g.start.max()),
             "buy_won_share": float(g.buy_won.mean()),
             "always_buy": float(g.buy_won.mean()), "always_rent": float(1 - g.buy_won.mean()),
             "estimate": float((g.est_buy == g.buy_won).mean()),
             "mean_reversion": float((g.mr_buy == g.buy_won).mean())}
        if "net_buy" in g:
            s["net"] = float((g.net_buy == g.buy_won).mean())
            ind = g[g.net_clipped == 0]
            s["net_in_domain"] = {"n": int(len(ind)), "acc": float((ind.net_buy == ind.buy_won).mean()) if len(ind) else None}
        for a, b, nm in ((1950, 1979, "1950-79"), (1980, 1999, "1980-99"), (2000, 2020, "2000+")):
            k = g[(g.start >= a) & (g.start <= b)]
            if len(k):
                s.setdefault("by_period", {})[nm] = {"n": int(len(k)), "buy_won": float(k.buy_won.mean()),
                                                     "estimate": float((k.est_buy == k.buy_won).mean()),
                                                     "mean_reversion": float((k.mr_buy == k.buy_won).mean())}
        out[int(H)] = s
    return out


# ------------------------------------------------------------------ majetok v oboch scenároch pre zvolený rok
def scenario_wealth(annual: pd.DataFrame, start: int, horizon: int, price: float = 250_000, p=None) -> dict:
    """Majetok oboch domácností po `horizon` rokoch pre nákup v roku `start` (skutočné dáta), v eurách.

    nominal = v eurách roku start+horizon, real = v eurách roku `start` (po odpočítaní inflácie)."""
    buyer, renter, adv, defl = tier2_run(annual, start, horizon, parts=True, p=p)
    f = lambda x: float(x[-1] * price)
    return {"start": start, "horizon": horizon, "price": price,
            "nominal": {"hypoteka": f(buyer), "prenajom": f(renter), "rozdiel": f(adv)},
            "real": {"hypoteka": f(buyer / defl), "prenajom": f(renter / defl), "rozdiel": f(adv / defl)},
            "winner": "hypotéka" if adv[-1] > 0 else "prenájom"}


def advice_today(annual: pd.DataFrame, net: dict | None, years: int, year: int | None = None, overrides: dict | None = None) -> dict:
    """Rada pre dnešok: odhady z posledných LOOKBACK rokov (alebo vlastné `overrides`) -> obe odpovede."""
    d = annual.set_index("year").sort_index()
    inp = known_inputs(d, int(year or d.index.max()))
    inp.update(overrides or {})
    adv = advise_estimate(inp, years)
    out = {"inputs": inp, "years": years, "estimate_adv": adv, "estimate_answer": "hypotéka" if adv > 0 else "prenájom"}
    if net is not None:
        pb, clipped = advise_net(net, inp, years)
        out.update(net_p_buy=pb, net_answer="hypotéka" if pb > 0.5 else "prenájom", net_clipped_inputs=clipped)
    return out


net = load_net(json.loads(NN_JSON))

## A · Majetok v oboch scenároch pre zvolený počiatočný rok
Zmeň `START` (1950–2015), `HORIZON` (roky) a `PRICE` (cena bytu v €).

In [ ]:
START, HORIZON, PRICE = 1990, 10, 250_000

w = scenario_wealth(annual, START, HORIZON, PRICE)
print(f"Nákup v roku {START}, byt za {PRICE:,.0f} €, horizont {HORIZON} rokov")
print(f"{'':12}{'v eurách roku '+str(START+HORIZON):>22}{'v eurách roku '+str(START):>22}")
for k, nm in (('prenajom', 'Podnájom'), ('hypoteka', 'Hypotéka'), ('rozdiel', 'Rozdiel (H−P)')):
    print(f"{nm:12}{w['nominal'][k]:>20,.0f} €{w['real'][k]:>20,.0f} €")
print('Víťaz:', w['winner'].upper())

## B · Rada modelu pre zadanú situáciu
`YEARS` = koľko rokov plánuješ bývať. Model si odhady (rast cien, nájmov, akcií, sadzbu) vezme z posledných 20 rokov dát; môžeš ich prepísať v `OVERRIDES` (napr. vlastný očakávaný rast cien).

In [ ]:
YEARS = 15
OVERRIDES = {}   # napr. {'house_growth': 0.04, 'gross_yield': 0.05, 'mortgage_rate': 0.03}

a = advice_today(annual, net, YEARS, overrides=OVERRIDES)
print('Vstupy modelu:', {k: round(v, 4) for k, v in a['inputs'].items()})
print(f"Model „odhad z histórie“: {a['estimate_answer'].upper()}  (výhoda hypotéky {a['estimate_adv']:+.2f} ceny bytu)")
print(f"Neurónová sieť:           {a['net_answer'].upper()}  (P(hypotéka vyhrá) = {a['net_p_buy']:.0%}; orezaných vstupov: {a['net_clipped_inputs']})")

## C · Vyhodnotenie na histórii
Pre každý počiatočný rok XY (v ktorom celé obdobie leží v dátach) sa spočíta, ktorý scenár skutočne vyhral, a porovná s odpoveďou modelu, ktorá využíva **len dáta do roku XY**. Tabuľka ukazuje podiel rokov, v ktorých mal model pravdu, spolu s jednoduchými stratégiami „vždy hypotéka“ a „vždy podnájom“.

In [ ]:
ev = evaluate(annual, net)
sm = summarize(ev)
print(f"{'horizont':>9}{'roky':>6}{'vždy hyp.':>11}{'vždy prenáj.':>14}{'odhad':>8}{'návrat k pr.':>14}{'sieť':>7}")
for H, s in sm.items():
    print(f"{H:>7} r.{s['n_years']:>6}{s['always_buy']:>11.0%}{s['always_rent']:>14.0%}{s['estimate']:>8.0%}{s['mean_reversion']:>14.0%}{s['net']:>7.0%}")
print()
print("Sieť dostane tie isté odhadnuté vstupy ako model „odhad z histórie“ (rozsahy vstupov pokrývajú aj vysokú infláciu).")

In [ ]:
# graf: skutočná výhoda hypotéky pre horizont 10 rokov a odpoveď modelu
import matplotlib.pyplot as plt
g = ev[ev.H == 10]
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.bar(g.start, g.actual_adv, color=['tab:orange' if b else 'tab:blue' for b in g.buy_won])
ok = (g.est_buy == g.buy_won)
ax.scatter(g.start[ok], [0]*ok.sum(), marker='o', s=14, color='green', label='odhad správne', zorder=3)
ax.scatter(g.start[~ok], [0]*(~ok).sum(), marker='x', s=22, color='red', label='odhad nesprávne', zorder=3)
ax.set_title('Výhoda hypotéky po 10 rokoch (oranžová = vyhrala hypotéka, modrá = podnájom)')
ax.set_xlabel('počiatočný rok'); ax.set_ylabel('podiel ceny (reálne)'); ax.legend(); plt.show()